# 0. Bootstrap NumPy / OpenCV for DA3

Run this cell before any runtime check or torch import.

DA3 requires `numpy<2`. New Colab images may start with NumPy 2.x. If NumPy 2 is imported before downgrade, the kernel must restart. This bootstrap cell pins NumPy/OpenCV, writes a marker file, and restarts the runtime once. After restart, run the notebook from the top again; the marker makes this cell skip the restart.

In [ ]:
# ============================================================
# 0. Bootstrap NumPy/OpenCV before importing torch or DA3
# ============================================================

from pathlib import Path
import os
import sys
import subprocess

BOOTSTRAP_MARKER = Path("/content/.da3_numpy_bootstrap_done")

if not BOOTSTRAP_MARKER.exists():
    print("First-time DA3 bootstrap: pinning NumPy/OpenCV before any heavy imports.")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--force-reinstall",
        "--no-cache-dir",
        "numpy==1.26.4",
        "opencv-python==4.10.0.84",
        "opencv-python-headless==4.10.0.84",
    ])
    BOOTSTRAP_MARKER.write_text("done", encoding="utf-8")
    print("Bootstrap install complete.")
    print("Restarting runtime now so Python loads NumPy 1.26.4 cleanly...")
    os.kill(os.getpid(), 9)
else:
    print("DA3 NumPy/OpenCV bootstrap marker exists. Continuing without restart.")
    import numpy as np
    print(f"NumPy version after bootstrap: {np.__version__}")
    if int(np.__version__.split(".")[0]) >= 2:
        raise RuntimeError(
            "Bootstrap marker exists, but NumPy is still >=2. "
            "Delete /content/.da3_numpy_bootstrap_done, rerun this cell, and let it restart the runtime."
        )

# RGB Video -> DA3 -> gsplat -> 3DGS Gaussian PLY

This Google Colab Pro notebook converts a short RGB video into a 3D Gaussian Splatting result.

Fixed repositories:

- Depth Anything 3: https://github.com/ByteDance-Seed/depth-anything-3
- gsplat: https://github.com/nerfstudio-project/gsplat

Input:

- RGB video
- Maximum duration: 300 seconds
- Supported formats in later cells: `.mp4`, `.mov`, `.mkv`, `.avi`

Final output:

```text
outputs/{job_id}/result/gaussians.ply
```

Important:

```text
gaussians.ply is a 3DGS Gaussian PLY, not a triangle mesh PLY.
```

A mesh PLY usually stores vertices and faces. A 3DGS Gaussian PLY stores Gaussian primitive attributes such as position, scale, rotation, opacity, and color or spherical harmonics. Opening the final file as a regular mesh may not show the expected surface.

Run order:

1. Project overview
2. Runtime check
3. Google Drive mount
4. Config
5. Directory setup
6. Dependency installation
7. Input video preparation
8. Frame extraction
9. DA3 repository setup
10. DA3 inference adapter
11. DA3 inference execution
12. DA3 result validation
13. Initial point cloud generation
14. gsplat repository setup
15. DA3-to-gsplat dataset conversion
16. gsplat training
17. Final Gaussian PLY export / download / debug utilities

In [ ]:
# ============================================================
# 2. Runtime check
# ============================================================

import sys
import os
import json
import subprocess
import shutil
from pathlib import Path
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from typing import Optional, Literal, Dict, Any, List

print("Python version:")
print(sys.version)
print()

try:
    import torch
    print("torch installed: True")
    print(f"torch version: {torch.__version__}")
    print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version reported by torch: {torch.version.cuda}")
        print(f"GPU count: {torch.cuda.device_count()}")
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU name: None")
except Exception as exc:
    print("torch installed: False or failed to import")
    print(f"torch import error: {repr(exc)}")

print()
print("nvidia-smi:")
!nvidia-smi

# 4. Config

All major settings are managed by `PipelineConfig`.

Drive default:

```text
/content/drive/MyDrive/da3_3dgs_colab
```

Local default:

```text
/content/da3_3dgs_colab
```

In [ ]:
# ============================================================
# 4. Config
# ============================================================

DA3_REPO_URL = "https://github.com/ByteDance-Seed/depth-anything-3"
GSPLAT_REPO_URL = "https://github.com/nerfstudio-project/gsplat"


@dataclass
class PipelineConfig:
    use_drive: bool = True
    video_source: Literal["upload", "drive_path"] = "drive_path"
    input_video_path: str = "/content/drive/MyDrive/da3_3dgs_colab/inputs/input_video.mp4"
    max_video_seconds: int = 300

    project_root: Optional[str] = None
    job_id: str = field(default_factory=lambda: datetime.now().strftime("%Y%m%d_%H%M%S"))

    output_dir: str = ""
    work_dir: str = ""
    frames_dir: str = ""
    da3_output_dir: str = ""
    pointcloud_dir: str = ""
    gsplat_dataset_dir: str = ""
    gsplat_train_dir: str = ""
    result_dir: str = ""
    result_ply_path: str = ""

    frame_fps: float = 2.0
    max_frames: int = 600
    resize_width: Optional[int] = 1008
    resize_height: Optional[int] = 756
    overwrite: bool = False
    resume: bool = True

    da3_repo_url: str = DA3_REPO_URL
    da3_repo_dir: str = "/content/depth-anything-3"
    da3_repo_revision: str = ""
    da3_model_name: str = "DepthAnything3"
    da3_checkpoint_or_hf_id: str = "depth-anything/DA3NESTED-GIANT-LARGE"
    da3_device: str = "cuda"
    da3_use_ray_pose: bool = False
    da3_mock_mode: bool = False
    confidence_threshold: float = 0.5

    point_stride: int = 4
    max_points: int = 1_000_000
    voxel_size: float = 0.01
    extrinsics_type: Literal["w2c", "c2w"] = "w2c"
    camera_convention: str = "opencv"

    gsplat_repo_url: str = GSPLAT_REPO_URL
    gsplat_repo_revision: str = ""
    repo_update_policy: Literal["reuse_existing", "pull_latest"] = "reuse_existing"
    install_gsplat_from_pip: bool = False
    gsplat_repo_dir: str = "/content/gsplat"
    gsplat_iterations: int = 7000
    # Training resume policy: current gsplat simple_trainer checkpoints do not
    # include enough optimizer/strategy state for guaranteed exact training resume.
    # - restart_if_incomplete: if ckpt exists but no Gaussian PLY, restart training safely.
    # - error_if_incomplete: stop and ask the user to decide.
    gsplat_resume_policy: Literal["restart_if_incomplete", "error_if_incomplete"] = "restart_if_incomplete"
    gsplat_final_train_ply: str = ""
    gsplat_command_template: str = (
        "python {repo_dir}/examples/da3_trainer.py "
        "--dataset-dir {dataset_dir} "
        "--train-dir {train_dir} "
        "--iterations {iterations} "
        "--points-ply {points_ply} "
        "--final-ply {result_ply}"
    )
    dry_run: bool = True

    metadata: Dict[str, Any] = field(default_factory=lambda: {
        "ply_note": "This is a 3DGS Gaussian PLY, not a triangle mesh PLY.",
        "da3_repo_url": DA3_REPO_URL,
        "gsplat_repo_url": GSPLAT_REPO_URL,
    })

    def __post_init__(self):
        if self.project_root is None:
            self.project_root = (
                "/content/drive/MyDrive/da3_3dgs_colab"
                if self.use_drive
                else "/content/da3_3dgs_colab"
            )
        self.refresh_paths()
        if self.da3_repo_url != DA3_REPO_URL:
            raise ValueError(f"da3_repo_url must be {DA3_REPO_URL}")
        if self.gsplat_repo_url != GSPLAT_REPO_URL:
            raise ValueError(f"gsplat_repo_url must be {GSPLAT_REPO_URL}")

    def refresh_paths(self):
        base = Path(self.project_root) / "outputs" / self.job_id
        self.output_dir = str(base)
        self.work_dir = str(base / "work")
        self.frames_dir = str(base / "work" / "frames")
        self.da3_output_dir = str(base / "work" / "da3")
        self.pointcloud_dir = str(base / "pointcloud")
        self.gsplat_dataset_dir = str(base / "work" / "gsplat_dataset")
        self.gsplat_train_dir = str(base / "work" / "gsplat_train")
        self.result_dir = str(base / "result")
        self.result_ply_path = str(base / "result" / "gaussians.ply")
        self.gsplat_final_train_ply = str(base / "work" / "gsplat_train" / "ply" / "gaussians.ply")


def print_config(config: PipelineConfig):
    data = asdict(config)
    key_width = max(len(k) for k in data)
    value_width = 100
    print("=" * (key_width + value_width + 7))
    print(f"| {'CONFIG KEY'.ljust(key_width)} | {'VALUE'.ljust(value_width)} |")
    print("=" * (key_width + value_width + 7))
    for key, value in data.items():
        value_str = json.dumps(value, ensure_ascii=False) if isinstance(value, dict) else str(value)
        chunks = [value_str[i:i + value_width] for i in range(0, len(value_str), value_width)] or [""]
        print(f"| {key.ljust(key_width)} | {chunks[0].ljust(value_width)} |")
        for chunk in chunks[1:]:
            print(f"| {' '.ljust(key_width)} | {chunk.ljust(value_width)} |")
    print("=" * (key_width + value_width + 7))


config = PipelineConfig(
    use_drive=True,
    video_source="drive_path",
    input_video_path="/content/drive/MyDrive/da3_3dgs_colab/inputs/input_video.mp4",
    # First-pass A100 smoke validation: enough to verify every stage and final PLY export
    # while spending minimal Colab Pro compute. Increase these after this run succeeds.
    job_id="smoke_a100_001",
    frame_fps=0.25,
    max_frames=12,
    resize_width=504,
    resize_height=378,
    overwrite=False,
    resume=True,
    da3_mock_mode=False,
    gsplat_iterations=50,
    dry_run=False,
)

print_config(config)
print(config.metadata["ply_note"])

In [ ]:
# ============================================================
# 3. Google Drive mount
# ============================================================

if config.use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    config.project_root = "/content/drive/MyDrive/da3_3dgs_colab"
    print("Google Drive mounted at /content/drive")
else:
    config.project_root = "/content/da3_3dgs_colab"
    print("Using local /content storage. This does not survive runtime reset.")

config.refresh_paths()
print(f"Project root: {config.project_root}")
print(f"Output dir: {config.output_dir}")
print(f"Final Gaussian PLY path: {config.result_ply_path}")

# 3.1 Drive Cache for Hugging Face and Torch

This cell stores Hugging Face and Torch caches on Google Drive so DA3 checkpoints do not need to be downloaded again every session.

Run this after Google Drive mount and before DA3 model loading.

In [ ]:
# ============================================================
# 3.1 Drive cache for Hugging Face and Torch
# Run after Google Drive mount and before DA3 setup/model loading.
# ============================================================

from pathlib import Path
import os

if config.use_drive:
    CACHE_ROOT = Path(config.project_root) / "cache"
else:
    CACHE_ROOT = Path("/content/da3_3dgs_colab/cache")

HF_HOME = CACHE_ROOT / "huggingface"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
TORCH_HOME = CACHE_ROOT / "torch"

for path in [HF_HOME, HF_HUB_CACHE, TRANSFORMERS_CACHE, TORCH_HOME]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)
os.environ["TORCH_HOME"] = str(TORCH_HOME)

print("Cache paths:")
print("  HF_HOME:", os.environ["HF_HOME"])
print("  HUGGINGFACE_HUB_CACHE:", os.environ["HUGGINGFACE_HUB_CACHE"])
print("  TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])
print("  TORCH_HOME:", os.environ["TORCH_HOME"])
print()
print("DA3 checkpoints downloaded by Hugging Face should be reused from this Drive cache in later sessions.")

In [ ]:
# ============================================================
# 5. Directory setup
# ============================================================

directories = [
    Path(config.output_dir),
    Path(config.work_dir),
    Path(config.frames_dir),
    Path(config.da3_output_dir),
    Path(config.da3_output_dir) / "depth",
    Path(config.da3_output_dir) / "confidence",
    Path(config.pointcloud_dir),
    Path(config.gsplat_dataset_dir),
    Path(config.gsplat_dataset_dir) / "images",
    Path(config.gsplat_train_dir),
    Path(config.result_dir),
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"created/exists: {directory}")

with open(Path(config.output_dir) / "config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(config), f, indent=2, ensure_ascii=False)

with open(Path(config.output_dir) / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(config.metadata, f, indent=2, ensure_ascii=False)

print("Directory setup complete.")
print(config.metadata["ply_note"])

# 6. Dependency Installation

Only base dependencies are installed here.

This cell does not reinstall torch. Colab's default torch is used first.

DA3 and gsplat installation are handled in later sections.

In [ ]:
# ============================================================
# 6. Dependency installation
# Base packages only. Do not reinstall torch here.
#
# IMPORTANT:
# DA3 currently requires numpy<2. New Colab images often start with NumPy 2.x.
# This cell pins NumPy/OpenCV at both the beginning and end of installation.
# After this cell finishes, restart the runtime before running the import check.
# ============================================================

# First pin NumPy/OpenCV to DA3-compatible versions.
!pip install -q --force-reinstall --no-cache-dir \
    "numpy==1.26.4" \
    "opencv-python==4.10.0.84" \
    "opencv-python-headless==4.10.0.84"

# Install the rest of the base dependencies.
!pip install -q \
    pillow \
    tqdm \
    matplotlib \
    open3d \
    plyfile \
    pyyaml \
    imageio \
    imageio-ffmpeg \
    scipy

# Re-pin again at the end in case any dependency resolver upgraded NumPy/OpenCV.
!pip install -q --force-reinstall --no-cache-dir \
    "numpy==1.26.4" \
    "opencv-python==4.10.0.84" \
    "opencv-python-headless==4.10.0.84"

print("Base dependency installation finished. torch was not reinstalled.")
print("NumPy/OpenCV were pinned to DA3-compatible versions:")
print("  numpy==1.26.4")
print("  opencv-python==4.10.0.84")
print("  opencv-python-headless==4.10.0.84")
print()
print("IMPORTANT NEXT STEP:")
print("  Runtime > Restart runtime")
print("Then run from the top again and continue to the import check.")
print("If you do not restart, Python may still use an old NumPy already loaded in memory.")
print("Resolver warnings for unused Colab packages requiring NumPy 2 can be ignored if the import check passes.")


In [ ]:
# ============================================================
# 6. Dependency import check
# ============================================================

import importlib

IMPORT_CHECKS = {
    "numpy": "numpy",
    "cv2": "opencv-python",
    "PIL": "pillow",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "open3d": "open3d",
    "plyfile": "plyfile",
    "yaml": "pyyaml",
    "imageio": "imageio",
    "imageio_ffmpeg": "imageio-ffmpeg",
    "scipy": "scipy",
}

failed = []
for module_name, package_name in IMPORT_CHECKS.items():
    try:
        module = importlib.import_module(module_name)
        print(f"[OK] {module_name} ({package_name}) {getattr(module, '__version__', '')}")
    except Exception as exc:
        print(f"[FAIL] {module_name} ({package_name}): {repr(exc)}")
        failed.append((module_name, package_name, repr(exc)))

try:
    import torch
    print(f"[OK] torch {torch.__version__}, cuda={torch.cuda.is_available()}")
except Exception as exc:
    print(f"[WARN] torch import failed: {repr(exc)}")

import numpy as _np
if int(_np.__version__.split(".")[0]) >= 2:
    raise RuntimeError(f"NumPy {_np.__version__} is installed, but DA3 requirements need numpy<2. Restart runtime, rerun dependency installation, and confirm numpy==1.26.4 before DA3 setup.")

if failed:
    raise ImportError(f"Base dependency import check failed: {failed}")

# 7. Input Video Preparation

Prepares the input video from Drive path or Colab upload, validates duration, and writes `video_metadata.json`.

In [ ]:
# ============================================================
# 7. Input video preparation
# ============================================================

import cv2

SUPPORTED_VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi"}


def prepare_input_video(config):
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if config.video_source == "upload":
        from google.colab import files
        uploaded = files.upload()
        if not uploaded or len(uploaded) != 1:
            raise ValueError("Upload exactly one video file.")
        source_path = Path("/content") / next(iter(uploaded.keys()))
    elif config.video_source == "drive_path":
        source_path = Path(config.input_video_path)
    else:
        raise ValueError("config.video_source must be 'upload' or 'drive_path'.")

    if not source_path.exists():
        raise FileNotFoundError(f"Input video does not exist: {source_path}")
    if source_path.suffix.lower() not in SUPPORTED_VIDEO_EXTENSIONS:
        raise ValueError(f"Unsupported extension: {source_path.suffix}")

    prepared_video_path = output_dir / f"input_video{source_path.suffix.lower()}"
    if source_path.resolve() != prepared_video_path.resolve():
        if not prepared_video_path.exists() or config.overwrite:
            shutil.copy2(source_path, prepared_video_path)
    cap = cv2.VideoCapture(str(prepared_video_path))
    if not cap.isOpened():
        raise ValueError(f"OpenCV could not open video: {prepared_video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if fps <= 0 or frame_count <= 0:
        raise ValueError(f"Could not compute duration: fps={fps}, frame_count={frame_count}")
    duration_sec = frame_count / fps
    if duration_sec > config.max_video_seconds:
        raise ValueError(f"Video duration {duration_sec:.2f}s exceeds {config.max_video_seconds}s.")

    metadata = {
        "source_path": str(source_path),
        "prepared_video_path": str(prepared_video_path),
        "extension": source_path.suffix.lower(),
        "fps": fps,
        "frame_count": frame_count,
        "duration_sec": duration_sec,
        "width": width,
        "height": height,
        "opencv_color_order": "BGR",
    }
    with open(output_dir / "video_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    print(json.dumps(metadata, indent=2))
    return prepared_video_path, metadata


input_video_path, video_metadata = prepare_input_video(config)

# 8. Frame Extraction

Extracts RGB frames to `work/frames/frame_000001.jpg` and writes `frames.json`.

In [ ]:
# ============================================================
# 8. Frame extraction
# Robust version: avoids the often-unreadable final video frame and
# falls back to nearby frames if OpenCV cannot decode the requested index.
# ============================================================

import math
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt


def read_frame_with_fallback(cap, target_index, source_frame_count, search_radius=8):
    """Read a frame robustly, trying nearby frames when the requested one fails."""
    target_index = int(target_index)
    source_frame_count = int(source_frame_count)

    offsets = [0]
    for delta in range(1, search_radius + 1):
        offsets.extend([-delta, delta])

    tried = []
    for offset in offsets:
        idx = target_index + offset
        if idx < 0 or idx >= source_frame_count:
            continue
        tried.append(idx)
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, bgr = cap.read()
        if ok and bgr is not None:
            return idx, bgr

    raise ValueError(
        f"Failed to read frame {target_index} and nearby frames {tried}. "
        "The video may be truncated/corrupted around this position."
    )


def extract_video_frames(config, input_video_path):
    frames_dir = Path(config.frames_dir)
    frames_dir.mkdir(parents=True, exist_ok=True)
    frames_json_path = Path(config.output_dir) / "frames.json"

    if config.resume and frames_json_path.exists() and not config.overwrite:
        print(f"Skipping frame extraction, using {frames_json_path}")
        return json.loads(frames_json_path.read_text(encoding="utf-8"))

    cap = cv2.VideoCapture(str(input_video_path))
    if not cap.isOpened():
        raise ValueError(f"OpenCV could not open video: {input_video_path}")

    source_fps = float(cap.get(cv2.CAP_PROP_FPS))
    source_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if source_fps <= 0 or source_frame_count <= 0:
        cap.release()
        raise ValueError("Invalid video metadata for extraction.")

    duration_sec = source_frame_count / source_fps
    requested_count = max(1, int(math.floor(duration_sec * config.frame_fps)))

    # Avoid sampling the exact final frame. Some mp4/mov files report a final
    # frame index that OpenCV cannot decode even when the rest of the video is fine.
    max_sample_index = max(0, source_frame_count - 2)
    indices = np.linspace(0, max_sample_index, num=requested_count, dtype=np.int64)

    if config.max_frames and len(indices) > config.max_frames:
        keep = np.linspace(0, len(indices) - 1, num=config.max_frames, dtype=np.int64)
        indices = indices[keep]

    indices = np.unique(indices)

    print("Frame extraction plan:")
    print(f"  source_fps: {source_fps}")
    print(f"  source_frame_count: {source_frame_count}")
    print(f"  duration_sec: {duration_sec:.3f}")
    print(f"  max_sample_index: {max_sample_index}")
    print(f"  requested_count before max_frames: {requested_count}")
    print(f"  frames_to_extract: {len(indices)}")

    frames = []
    fallback_count = 0

    for output_index, requested_frame_index in enumerate(tqdm(indices, desc="Extracting frames"), start=1):
        actual_frame_index, bgr = read_frame_with_fallback(
            cap,
            requested_frame_index,
            source_frame_count,
            search_radius=8,
        )
        if actual_frame_index != int(requested_frame_index):
            fallback_count += 1
            print(
                f"[WARN] Requested frame {int(requested_frame_index)} was unreadable; "
                f"used nearby frame {actual_frame_index}."
            )

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        if config.resize_width and config.resize_height:
            rgb = cv2.resize(
                rgb,
                (int(config.resize_width), int(config.resize_height)),
                interpolation=cv2.INTER_AREA,
            )

        height, width = rgb.shape[:2]
        image_path = frames_dir / f"frame_{output_index:06d}.jpg"
        Image.fromarray(rgb).save(image_path, quality=95)

        frames.append({
            "image_path": str(image_path),
            "original_frame_index": int(actual_frame_index),
            "requested_frame_index": int(requested_frame_index),
            "timestamp_sec": float(actual_frame_index) / source_fps,
            "width": int(width),
            "height": int(height),
        })

    cap.release()

    if not frames:
        raise ValueError("No frames were extracted. Check video readability and sampling config.")

    manifest = {
        "frames": frames,
        "source_fps": source_fps,
        "source_frame_count": source_frame_count,
        "duration_sec": duration_sec,
        "fallback_read_count": fallback_count,
        "note": "original_frame_index is the actually decoded frame. requested_frame_index is the sampled target.",
    }

    # Keep backward compatibility: most later cells expect frames.json to be a list.
    # So save the list at frames.json and a richer manifest separately.
    frames_json_path.write_text(json.dumps(frames, indent=2, ensure_ascii=False), encoding="utf-8")
    (Path(config.output_dir) / "frames_manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Saved {len(frames)} frames and {frames_json_path}")
    print(f"Saved detailed frame manifest: {Path(config.output_dir) / 'frames_manifest.json'}")
    print(f"Fallback frame reads: {fallback_count}")
    return frames


def preview_sample_frames(frames, count=4):
    sample_count = min(count, len(frames))
    ids = np.linspace(0, len(frames) - 1, num=sample_count, dtype=np.int64)
    fig, axes = plt.subplots(1, sample_count, figsize=(4 * sample_count, 4))
    if sample_count == 1:
        axes = [axes]
    for ax, idx in zip(axes, ids):
        item = frames[int(idx)]
        ax.imshow(Image.open(item["image_path"]).convert("RGB"))
        ax.set_title(f"frame {int(idx)+1}\nt={item['timestamp_sec']:.2f}s")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


frames = extract_video_frames(config, input_video_path)
preview_sample_frames(frames)


# 9. Depth Anything 3 Repository Setup

Clones and installs the official DA3 repository, then verifies the official import:

```python
from depth_anything_3.api import DepthAnything3
```

In [ ]:
# ============================================================
# 9. DA3 repository setup
# ============================================================

import traceback
import numpy as _np
if int(_np.__version__.split(".")[0]) >= 2:
    raise RuntimeError(f"NumPy {_np.__version__} is loaded. DA3 requires numpy<2. Restart runtime, rerun section 6 with numpy==1.26.4, then continue.")
print(f"NumPy version before DA3 setup: {_np.__version__}")

EXPECTED_DA3_REPO_URL = "https://github.com/ByteDance-Seed/depth-anything-3"
DA3_REPO_DIR = Path(config.da3_repo_dir)


def run_shell(command, cwd=None, check=True):
    print(f"\n$ {command}")
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {command}")
    return result


def setup_da3_repository(config):
    if config.da3_repo_url != EXPECTED_DA3_REPO_URL:
        raise ValueError(f"DA3 repo must be {EXPECTED_DA3_REPO_URL}")
    if DA3_REPO_DIR.exists():
        print(f"DA3 repo exists: {DA3_REPO_DIR}")
        if (DA3_REPO_DIR / ".git").exists() and config.repo_update_policy == "pull_latest":
            run_shell("git pull", cwd=DA3_REPO_DIR, check=False)
        else:
            print("Repo update policy is reuse_existing; skipping git pull for reproducibility.")
    else:
        run_shell(f"git clone {config.da3_repo_url} {DA3_REPO_DIR}", cwd="/content")

    if config.da3_repo_revision and (DA3_REPO_DIR / ".git").exists():
        run_shell(f"git checkout {config.da3_repo_revision}", cwd=DA3_REPO_DIR)

    commit_result = run_shell("git rev-parse HEAD", cwd=DA3_REPO_DIR, check=False) if (DA3_REPO_DIR / ".git").exists() else None
    da3_commit = commit_result.stdout.strip() if commit_result and commit_result.returncode == 0 else "unknown"

    structure = {
        "repo_dir": str(DA3_REPO_DIR),
        "src_depth_anything_3_exists": (DA3_REPO_DIR / "src" / "depth_anything_3").exists(),
        "notebooks_exists": (DA3_REPO_DIR / "notebooks").exists(),
        "requirements_txt_exists": (DA3_REPO_DIR / "requirements.txt").exists(),
        "pyproject_toml_exists": (DA3_REPO_DIR / "pyproject.toml").exists(),
        "git_commit": da3_commit,
        "repo_update_policy": config.repo_update_policy,
        "requested_revision": config.da3_repo_revision,
    }
    print(json.dumps(structure, indent=2))
    (Path(config.output_dir) / "da3_repository_setup.json").write_text(json.dumps(structure, indent=2), encoding="utf-8")

    if structure["requirements_txt_exists"]:
        run_shell(f"{sys.executable} -m pip install -r requirements.txt", cwd=DA3_REPO_DIR)
        # DA3 requires numpy<2. On newer Colab images, unpinned OpenCV packages may pull NumPy 2 again.
        # Re-pin the runtime-critical pair before importing DA3.
        run_shell(f"{sys.executable} -m pip install 'numpy==1.26.4' 'opencv-python==4.10.0.84' 'opencv-python-headless==4.10.0.84'", cwd=DA3_REPO_DIR)
        print("If pip changed numpy during this step, restart the runtime and rerun from the top before importing DA3.")
    if structure["pyproject_toml_exists"]:
        run_shell(f"{sys.executable} -m pip install -e .", cwd=DA3_REPO_DIR)

    for candidate in [DA3_REPO_DIR / "src", DA3_REPO_DIR]:
        if candidate.exists() and str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

    try:
        from depth_anything_3.api import DepthAnything3
    except Exception as exc:
        traceback.print_exc()
        raise RuntimeError("DA3 official API import failed. Do not invent a replacement API.") from exc
    if not hasattr(DepthAnything3, "from_pretrained"):
        raise RuntimeError("DepthAnything3.from_pretrained is missing. TODO: inspect DA3 API.")
    print("[OK] DA3 official API import and from_pretrained verified.")
    return structure


da3_repo_structure = setup_da3_repository(config)

# 10. DA3 Inference Adapter

Defines `DA3Adapter` and `DA3Result`. Mock mode is supported with synthetic depth, confidence, intrinsics, and w2c extrinsics.

In [ ]:
# ============================================================
# 10. DA3 inference adapter
# ============================================================

from typing import List, Optional


@dataclass
class DA3Result:
    processed_image_paths: List[str]
    intrinsics: np.ndarray
    extrinsics: np.ndarray
    width: int
    height: int
    extrinsics_type: str
    camera_convention: str
    depth: Optional[np.ndarray] = None
    confidence: Optional[np.ndarray] = None
    depth_paths: List[str] = field(default_factory=list)
    confidence_paths: List[str] = field(default_factory=list)


class DA3Adapter:
    def __init__(self, config):
        self.config = config
        self.model = None
        self.output_dir = Path(config.da3_output_dir)
        self.depth_dir = self.output_dir / "depth"
        self.confidence_dir = self.output_dir / "confidence"
        self.cameras_json_path = self.output_dir / "cameras.json"
        self.depth_dir.mkdir(parents=True, exist_ok=True)
        self.confidence_dir.mkdir(parents=True, exist_ok=True)

    def load_model(self):
        if self.config.da3_mock_mode:
            print("DA3 mock mode enabled. Skipping real model load.")
            return None
        try:
            from depth_anything_3.api import DepthAnything3
        except Exception as exc:
            raise RuntimeError("Failed to import depth_anything_3.api.DepthAnything3") from exc
        if not hasattr(DepthAnything3, "from_pretrained"):
            raise RuntimeError("DepthAnything3.from_pretrained missing. TODO: inspect DA3 API.")
        self.model = DepthAnything3.from_pretrained(self.config.da3_checkpoint_or_hf_id)
        if hasattr(self.model, "to"):
            self.model = self.model.to(self.config.da3_device)
        if not hasattr(self.model, "inference"):
            raise RuntimeError("Loaded DA3 model has no inference(images). TODO: inspect DA3 API.")
        return self.model

    def infer(self, image_paths: List[Path]) -> DA3Result:
        image_paths = [Path(p) for p in image_paths]
        if self.config.da3_mock_mode:
            return self._mock_infer(image_paths)
        if self.model is None:
            self.load_model()
        images = [Image.open(p).convert("RGB") for p in tqdm(image_paths, desc="Loading DA3 images")]
        prediction = self.model.inference(images)
        self._validate_prediction(prediction, len(images))
        width, height = images[0].size
        return DA3Result(
            processed_image_paths=[str(p) for p in image_paths],
            depth=np.asarray(prediction.depth, dtype=np.float32),
            confidence=np.asarray(prediction.conf, dtype=np.float32),
            intrinsics=np.asarray(prediction.intrinsics, dtype=np.float32),
            extrinsics=np.asarray(prediction.extrinsics, dtype=np.float32),
            width=int(width),
            height=int(height),
            extrinsics_type="w2c",
            camera_convention="opencv",
        )

    def save_outputs(self, result: DA3Result) -> DA3Result:
        if result.depth is None or result.confidence is None:
            raise ValueError("result.depth and result.confidence are required.")
        n = len(result.processed_image_paths)
        if result.extrinsics.shape != (n, 3, 4):
            raise ValueError(f"Expected extrinsics [N,3,4], got {result.extrinsics.shape}")
        if result.intrinsics.shape != (n, 3, 3):
            raise ValueError(f"Expected intrinsics [N,3,3], got {result.intrinsics.shape}")
        frames_out = []
        result.depth_paths = []
        result.confidence_paths = []
        for idx in tqdm(range(n), desc="Saving DA3 outputs"):
            depth_path = self.depth_dir / f"depth_{idx+1:06d}.npy"
            conf_path = self.confidence_dir / f"conf_{idx+1:06d}.npy"
            np.save(depth_path, result.depth[idx].astype(np.float32))
            np.save(conf_path, result.confidence[idx].astype(np.float32))
            result.depth_paths.append(str(depth_path))
            result.confidence_paths.append(str(conf_path))
            frames_out.append({
                "image_path": result.processed_image_paths[idx],
                "depth_path": str(depth_path),
                "confidence_path": str(conf_path),
                "intrinsic": result.intrinsics[idx].astype(float).tolist(),
                "extrinsic": result.extrinsics[idx].astype(float).tolist(),
                "extrinsics_type": "w2c",
                "camera_convention": "opencv",
                "note": "DA3 README describes extrinsics as OpenCV w2c or COLMAP format.",
            })
        cameras_json = {
            "width": result.width,
            "height": result.height,
            "extrinsics_type": "w2c",
            "camera_convention": "opencv",
            "note": "DA3 README describes extrinsics as OpenCV w2c or COLMAP format.",
            "frames": frames_out,
        }
        self.cameras_json_path.write_text(json.dumps(cameras_json, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"Saved {self.cameras_json_path}")
        return result

    def _validate_prediction(self, prediction, expected_count):
        missing = [x for x in ["depth", "conf", "extrinsics", "intrinsics"] if not hasattr(prediction, x)]
        if missing:
            raise RuntimeError(f"DA3 prediction missing fields: {missing}")
        extrinsics = np.asarray(prediction.extrinsics)
        intrinsics = np.asarray(prediction.intrinsics)
        if extrinsics.shape != (expected_count, 3, 4):
            raise ValueError(f"Expected extrinsics [N,3,4], got {extrinsics.shape}")
        if intrinsics.shape != (expected_count, 3, 3):
            raise ValueError(f"Expected intrinsics [N,3,3], got {intrinsics.shape}")
        print("DA3 prediction validation passed.")

    def _mock_infer(self, image_paths):
        image = Image.open(image_paths[0]).convert("RGB")
        width, height = image.size
        n = len(image_paths)
        yy = np.linspace(0.4, 2.0, height, dtype=np.float32).reshape(height, 1)
        xx = np.linspace(0.0, 0.3, width, dtype=np.float32).reshape(1, width)
        depth = np.stack([yy + xx + 0.01 * i for i in range(n)], axis=0).astype(np.float32)
        confidence = np.ones((n, height, width), dtype=np.float32)
        fx = fy = float(max(width, height))
        cx = width * 0.5
        cy = height * 0.5
        intrinsics = np.tile(np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float32), (n, 1, 1))
        extrinsics = np.zeros((n, 3, 4), dtype=np.float32)
        for i in range(n):
            extrinsics[i] = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, -0.03 * i]], dtype=np.float32)
        return DA3Result([str(p) for p in image_paths], intrinsics, extrinsics, width, height, "w2c", "opencv", depth, confidence)

    def load_existing_result(self):
        cameras = json.loads(self.cameras_json_path.read_text(encoding="utf-8"))
        frames = cameras["frames"]
        return DA3Result(
            processed_image_paths=[x["image_path"] for x in frames],
            depth_paths=[x["depth_path"] for x in frames],
            confidence_paths=[x["confidence_path"] for x in frames],
            intrinsics=np.asarray([x["intrinsic"] for x in frames], dtype=np.float32),
            extrinsics=np.asarray([x["extrinsic"] for x in frames], dtype=np.float32),
            width=int(cameras["width"]),
            height=int(cameras["height"]),
            extrinsics_type=cameras.get("extrinsics_type", "w2c"),
            camera_convention=cameras.get("camera_convention", "opencv"),
        )

# 11. DA3 Inference Execution

Runs DA3 inference or resumes from existing `cameras.json`, depth files, and confidence files.

In [ ]:
# ============================================================
# 11. DA3 inference execution
# ============================================================


def check_da3_memory_budget(config, image_paths):
    """Fail early with actionable settings before DA3 OOMs on small GPUs."""
    if config.da3_mock_mode:
        return
    try:
        import torch
        if not torch.cuda.is_available():
            return
        props = torch.cuda.get_device_properties(0)
        total_gb = props.total_memory / (1024 ** 3)
    except Exception:
        return

    from PIL import Image
    with Image.open(image_paths[0]) as im:
        width, height = im.size
    pixels = width * height
    image_count = len(image_paths)

    print("DA3 memory budget check:")
    print(f"  GPU VRAM: {total_gb:.2f} GiB")
    print(f"  image_count: {image_count}")
    print(f"  image_size: {width}x{height}")
    print(f"  da3_model: {config.da3_checkpoint_or_hf_id}")

    if total_gb < 20 and (image_count > 24 or pixels > 504 * 378):
        message = (
            "DA3 inference is likely to OOM on this GPU. "
            f"Detected {total_gb:.2f} GiB VRAM, {image_count} frames, {width}x{height}.\n\n"
            "For T4 / ~15GB VRAM, use a small smoke-test config first:\n"
            "  frame_fps=0.25\n"
            "  max_frames=10 or 20\n"
            "  resize_width=504\n"
            "  resize_height=378\n"
            "  gsplat_iterations=50 or 200\n"
            "  da3_use_ray_pose=False\n\n"
            "For full quality, switch Colab runtime to A100/L4 high-memory or reduce frames/resolution."
        )
        raise RuntimeError(message)

def da3_outputs_complete(config, expected_count):
    da3_dir = Path(config.da3_output_dir)
    cameras_path = da3_dir / "cameras.json"
    if not cameras_path.exists():
        return False
    depth_files = sorted((da3_dir / "depth").glob("depth_*.npy"))
    conf_files = sorted((da3_dir / "confidence").glob("conf_*.npy"))
    if len(depth_files) != expected_count or len(conf_files) != expected_count:
        return False
    try:
        cameras = json.loads(cameras_path.read_text(encoding="utf-8"))
        return len(cameras.get("frames", [])) == expected_count
    except Exception:
        return False


try:
    frame_items = json.loads((Path(config.output_dir) / "frames.json").read_text(encoding="utf-8"))
    image_paths = [Path(item["image_path"]) for item in frame_items]
    print(f"DA3 mock mode: {config.da3_mock_mode}")
    print(f"DA3 checkpoint/HF ID: {config.da3_checkpoint_or_hf_id}")
    da3_adapter = DA3Adapter(config)
    if config.resume and not config.overwrite and da3_outputs_complete(config, len(image_paths)):
        print("Complete DA3 outputs exist. Skipping inference.")
        da3_result = da3_adapter.load_existing_result()
    else:
        da3_adapter.load_model()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        da3_result = da3_adapter.infer(image_paths)
        da3_result = da3_adapter.save_outputs(da3_result)
    print("DA3 inference execution completed.")
except Exception as exc:
    print("DA3 inference failed.")
    print("Check DA3 repo install, checkpoint/HF ID, depth_anything_3.api import, and prediction fields.")
    traceback.print_exc()
    raise

# 12. DA3 Result Validation

Validates depth, confidence, and camera outputs, then previews sample depth maps.

In [ ]:
# ============================================================
# 12. DA3 result validation
# ============================================================

def validate_da3_results(config):
    frames = json.loads((Path(config.output_dir) / "frames.json").read_text(encoding="utf-8"))
    expected = len(frames)
    cameras_path = Path(config.da3_output_dir) / "cameras.json"
    cameras = json.loads(cameras_path.read_text(encoding="utf-8"))
    depth_files = sorted((Path(config.da3_output_dir) / "depth").glob("depth_*.npy"))
    conf_files = sorted((Path(config.da3_output_dir) / "confidence").glob("conf_*.npy"))
    if len(depth_files) != expected:
        raise ValueError(f"Depth count mismatch: {len(depth_files)} vs {expected}")
    if len(conf_files) != expected:
        raise ValueError(f"Confidence count mismatch: {len(conf_files)} vs {expected}")
    if len(cameras.get("frames", [])) != expected:
        raise ValueError(f"Camera count mismatch: {len(cameras.get('frames', []))} vs {expected}")
    for i, item in enumerate(cameras["frames"]):
        if np.asarray(item["intrinsic"]).shape != (3, 3):
            raise ValueError(f"Bad intrinsic shape at {i}")
        if np.asarray(item["extrinsic"]).shape != (3, 4):
            raise ValueError(f"Bad extrinsic shape at {i}")
        depth = np.load(depth_files[i])
        conf = np.load(conf_files[i])
        if depth.shape != conf.shape:
            raise ValueError(f"Depth/conf shape mismatch at {i}: {depth.shape} vs {conf.shape}")
    validation = {
        "status": "ok",
        "frame_count": expected,
        "depth_count": len(depth_files),
        "confidence_count": len(conf_files),
        "camera_count": len(cameras["frames"]),
        "da3_mock_mode": bool(config.da3_mock_mode),
    }
    (Path(config.da3_output_dir) / "da3_validation.json").write_text(json.dumps(validation, indent=2), encoding="utf-8")
    print(json.dumps(validation, indent=2))
    return validation


def preview_depth_maps(config, count=4):
    depth_files = sorted((Path(config.da3_output_dir) / "depth").glob("depth_*.npy"))
    sample_count = min(count, len(depth_files))
    ids = np.linspace(0, len(depth_files) - 1, sample_count, dtype=np.int64)
    fig, axes = plt.subplots(1, sample_count, figsize=(4 * sample_count, 4))
    if sample_count == 1:
        axes = [axes]
    for ax, idx in zip(axes, ids):
        depth = np.load(depth_files[int(idx)])
        valid = np.isfinite(depth)
        vmin, vmax = (np.percentile(depth[valid], 2), np.percentile(depth[valid], 98)) if valid.any() else (0, 1)
        ax.imshow(depth, cmap="magma", vmin=vmin, vmax=vmax)
        ax.set_title(depth_files[int(idx)].name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


try:
    da3_validation = validate_da3_results(config)
    preview_depth_maps(config)
except Exception:
    print("DA3 validation failed. Check DA3 repo install, checkpoint/HF ID, import, and prediction fields.")
    traceback.print_exc()
    raise

# 13. Initial Point Cloud Generation

Creates `outputs/{job_id}/pointcloud/init_points.ply` from DA3 depth and cameras.

This file is a regular point cloud PLY for initialization. It is not the final 3DGS Gaussian PLY.

In [ ]:
# ============================================================
# 13. Initial point cloud generation
# ============================================================

POINTCLOUD_OUTPUT_DIR = Path(config.pointcloud_dir)
POINTCLOUD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def extrinsic_3x4_to_c2w_4x4(extrinsic, extrinsics_type):
    mat = np.eye(4, dtype=np.float32)
    mat[:3, :4] = np.asarray(extrinsic, dtype=np.float32)
    if extrinsics_type == "w2c":
        return np.linalg.inv(mat).astype(np.float32)
    if extrinsics_type == "c2w":
        return mat.astype(np.float32)
    raise ValueError(f"Unsupported extrinsics_type={extrinsics_type}")


def unproject_depth_frame(depth, confidence, rgb, intrinsic, extrinsic, extrinsics_type, config):
    h, w = depth.shape
    if rgb.shape[:2] != (h, w):
        rgb = np.asarray(Image.fromarray(rgb).resize((w, h), resample=Image.BILINEAR), dtype=np.uint8)
    stride = max(1, int(config.point_stride))
    ys = np.arange(0, h, stride, dtype=np.int32)
    xs = np.arange(0, w, stride, dtype=np.int32)
    grid_x, grid_y = np.meshgrid(xs, ys)
    z = depth[grid_y, grid_x]
    conf = confidence[grid_y, grid_x]
    valid = np.isfinite(z) & (z > 0) & np.isfinite(conf) & (conf >= float(config.confidence_threshold))
    if not np.any(valid):
        return np.empty((0, 3), dtype=np.float32), np.empty((0, 3), dtype=np.uint8)
    u = grid_x[valid].astype(np.float32)
    v = grid_y[valid].astype(np.float32)
    z = z[valid].astype(np.float32)
    colors = rgb[grid_y[valid], grid_x[valid], :]
    K = np.asarray(intrinsic, dtype=np.float32)
    x = (u - K[0, 2]) / K[0, 0] * z
    y = (v - K[1, 2]) / K[1, 1] * z
    points_cam = np.stack([x, y, z], axis=1).astype(np.float32)
    c2w = extrinsic_3x4_to_c2w_4x4(extrinsic, extrinsics_type)
    points_h = np.concatenate([points_cam, np.ones((len(points_cam), 1), dtype=np.float32)], axis=1)
    points = (c2w @ points_h.T).T[:, :3].astype(np.float32)
    finite = np.isfinite(points).all(axis=1)
    return points[finite], colors[finite]


def generate_initial_point_cloud(config):
    import open3d as o3d
    init_ply_path = POINTCLOUD_OUTPUT_DIR / "init_points.ply"
    init_npz_path = POINTCLOUD_OUTPUT_DIR / "init_points.npz"
    if config.resume and init_ply_path.exists() and init_npz_path.exists() and not config.overwrite:
        print(f"Using existing init point cloud: {init_ply_path}")
        return init_ply_path
    cameras = json.loads((Path(config.da3_output_dir) / "cameras.json").read_text(encoding="utf-8"))
    all_points, all_colors, camera_centers = [], [], []
    for item in tqdm(cameras["frames"], desc="Backprojecting DA3 depth"):
        rgb = np.asarray(Image.open(item["image_path"]).convert("RGB"), dtype=np.uint8)
        depth = np.load(item["depth_path"]).astype(np.float32)
        conf = np.load(item["confidence_path"]).astype(np.float32)
        extrinsics_type = item.get("extrinsics_type", cameras.get("extrinsics_type", config.extrinsics_type))
        c2w = extrinsic_3x4_to_c2w_4x4(item["extrinsic"], extrinsics_type)
        camera_centers.append(c2w[:3, 3])
        points, colors = unproject_depth_frame(depth, conf, rgb, item["intrinsic"], item["extrinsic"], extrinsics_type, config)
        if len(points):
            all_points.append(points)
            all_colors.append(colors)
    if not all_points:
        raise ValueError("No valid points generated. Lower confidence_threshold or point_stride.")
    points = np.concatenate(all_points).astype(np.float32)
    colors = np.concatenate(all_colors).astype(np.uint8)
    before = len(points)
    if config.max_points and len(points) > int(config.max_points):
        rng = np.random.default_rng(42)
        idx = rng.choice(len(points), size=int(config.max_points), replace=False)
        points, colors = points[idx], colors[idx]
    after_sampling = len(points)
    if config.voxel_size and config.voxel_size > 0:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
        pcd.colors = o3d.utility.Vector3dVector((colors.astype(np.float32) / 255.0).astype(np.float64))
        pcd = pcd.voxel_down_sample(float(config.voxel_size))
        points = np.asarray(pcd.points, dtype=np.float32)
        colors = np.clip(np.asarray(pcd.colors) * 255, 0, 255).astype(np.uint8)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector((colors.astype(np.float32) / 255.0).astype(np.float64))
    o3d.io.write_point_cloud(str(init_ply_path), pcd, write_ascii=False)
    np.savez_compressed(init_npz_path, points=points, colors=colors)
    stats = {
        "note": "init_points.ply is a regular point cloud PLY for 3DGS initialization, not the final Gaussian PLY.",
        "points_before_sampling": int(before),
        "points_after_sampling": int(after_sampling),
        "points_after_voxel_downsample": int(len(points)),
        "init_ply_path": str(init_ply_path),
    }
    (POINTCLOUD_OUTPUT_DIR / "init_points_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
    print(json.dumps(stats, indent=2))
    print("This is NOT the final 3DGS Gaussian PLY.")
    visualize_camera_trajectory(np.asarray(camera_centers, dtype=np.float32))
    visualize_point_cloud_preview(points, colors)
    return init_ply_path


def visualize_camera_trajectory(camera_centers):
    if len(camera_centers) == 0:
        return
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(camera_centers[:, 0], camera_centers[:, 1], camera_centers[:, 2], marker="o")
    ax.set_title("Camera trajectory preview")
    plt.show()


def visualize_point_cloud_preview(points, colors, max_plot_points=20000):
    if len(points) > max_plot_points:
        idx = np.random.default_rng(42).choice(len(points), size=max_plot_points, replace=False)
        points, colors = points[idx], colors[idx]
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=np.clip(colors / 255.0, 0, 1), s=0.5)
    ax.set_title("Initial point cloud preview")
    plt.show()


init_pointcloud_path = generate_initial_point_cloud(config)

# 13.1 Quality and Coordinate Validation

This section validates DA3 camera conventions and point cloud scale before gsplat training.

Checks:

- DA3 `w2c` to `c2w` inversion consistency
- camera trajectory sanity
- initial point cloud bounding box and finite values
- scene scale warnings
- OpenCV camera convention reminder for gsplat

By default, this cell does not mutate the dataset. If `config.auto_normalize_scene=True`, it writes a separate normalized point cloud and records the transform, but the primary training path remains unchanged unless later cells are explicitly pointed to the normalized dataset.

In [ ]:
# ============================================================
# 13.1 Quality and coordinate validation
# ============================================================

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d


def _as_w2c_4x4(extrinsic, extrinsics_type):
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :4] = np.asarray(extrinsic, dtype=np.float64)
    if extrinsics_type == "w2c":
        return mat
    if extrinsics_type == "c2w":
        return np.linalg.inv(mat)
    raise ValueError(f"Unsupported extrinsics_type={extrinsics_type}")


def _as_c2w_4x4(extrinsic, extrinsics_type):
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :4] = np.asarray(extrinsic, dtype=np.float64)
    if extrinsics_type == "w2c":
        return np.linalg.inv(mat)
    if extrinsics_type == "c2w":
        return mat
    raise ValueError(f"Unsupported extrinsics_type={extrinsics_type}")


def validate_da3_camera_and_scene_quality(config):
    cameras_path = Path(config.da3_output_dir) / "cameras.json"
    init_points_path = Path(config.pointcloud_dir) / "init_points.ply"
    quality_dir = Path(config.output_dir) / "quality_checks"
    quality_dir.mkdir(parents=True, exist_ok=True)

    if not cameras_path.exists():
        raise FileNotFoundError(cameras_path)
    if not init_points_path.exists():
        raise FileNotFoundError(init_points_path)

    cameras = json.loads(cameras_path.read_text(encoding="utf-8"))
    frames = cameras.get("frames", [])
    if not frames:
        raise ValueError("No camera frames found in cameras.json")

    c2ws = []
    w2c_roundtrip_errors = []
    det_errors = []
    forward_axes = []

    for item in frames:
        ext_type = item.get("extrinsics_type", cameras.get("extrinsics_type", config.extrinsics_type))
        w2c = _as_w2c_4x4(item["extrinsic"], ext_type)
        c2w = _as_c2w_4x4(item["extrinsic"], ext_type)
        identity_error = float(np.max(np.abs(w2c @ c2w - np.eye(4))))
        w2c_roundtrip_errors.append(identity_error)
        R = c2w[:3, :3]
        det_errors.append(float(abs(np.linalg.det(R) - 1.0)))
        # OpenCV camera looks along +Z in camera coordinates; c2w maps that to world.
        forward_axes.append(R[:, 2])
        c2ws.append(c2w)

    c2ws = np.stack(c2ws, axis=0)
    camera_centers = c2ws[:, :3, 3]
    forward_axes = np.stack(forward_axes, axis=0)

    pcd = o3d.io.read_point_cloud(str(init_points_path))
    points = np.asarray(pcd.points, dtype=np.float64)
    colors = np.asarray(pcd.colors, dtype=np.float64)
    if len(points) == 0:
        raise ValueError("init_points.ply contains no points")
    finite_mask = np.isfinite(points).all(axis=1)
    finite_ratio = float(finite_mask.mean())
    points_finite = points[finite_mask]

    bbox_min = points_finite.min(axis=0)
    bbox_max = points_finite.max(axis=0)
    bbox_extent = bbox_max - bbox_min
    bbox_diag = float(np.linalg.norm(bbox_extent))
    camera_extent = float(np.linalg.norm(camera_centers - camera_centers.mean(axis=0), axis=1).max()) if len(camera_centers) > 1 else 0.0

    warnings = []
    if max(w2c_roundtrip_errors) > 1e-3:
        warnings.append("w2c/c2w roundtrip error is high. Check extrinsics_type.")
    if max(det_errors) > 1e-3:
        warnings.append("Camera rotation determinant deviates from 1. Check extrinsics matrix validity.")
    if finite_ratio < 0.999:
        warnings.append("Point cloud contains non-finite points.")
    scene_extent_warn_threshold = float(getattr(config, "scene_extent_warn_threshold", 1000.0))
    scene_extent_error_threshold = float(getattr(config, "scene_extent_error_threshold", 1000000.0))
    auto_normalize_scene = bool(getattr(config, "auto_normalize_scene", False))

    if bbox_diag > scene_extent_warn_threshold:
        warnings.append("Point cloud bbox is very large. Consider checking DA3 depth scale or enabling controlled normalization.")
    if bbox_diag > scene_extent_error_threshold:
        warnings.append("Point cloud bbox is extremely large. Training may be unstable without normalization.")
    if config.camera_convention != "opencv":
        warnings.append("This pipeline expects OpenCV camera convention. Current config.camera_convention differs.")

    report = {
        "camera_count": int(len(frames)),
        "extrinsics_type_top_level": cameras.get("extrinsics_type", None),
        "config_extrinsics_type": config.extrinsics_type,
        "camera_convention": config.camera_convention,
        "opencv_convention_note": "OpenCV camera coordinates are assumed. Camera forward is +Z before c2w transform.",
        "max_w2c_c2w_roundtrip_error": float(max(w2c_roundtrip_errors)),
        "max_rotation_det_error": float(max(det_errors)),
        "camera_extent": camera_extent,
        "point_count": int(len(points)),
        "finite_point_ratio": finite_ratio,
        "bbox_min": bbox_min.tolist(),
        "bbox_max": bbox_max.tolist(),
        "bbox_extent": bbox_extent.tolist(),
        "bbox_diag": bbox_diag,
        "scene_extent_warn_threshold": scene_extent_warn_threshold,
        "scene_extent_error_threshold": scene_extent_error_threshold,
        "auto_normalize_scene": auto_normalize_scene,
        "warnings": warnings,
        "training_note": (
            "Primary training keeps DA3 scale unchanged. If scale is unreasonable, inspect this report before enabling normalization."
        ),
    }

    report_path = quality_dir / "camera_scene_quality_report.json"
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

    print("Camera / scene quality report:")
    print(json.dumps(report, indent=2, ensure_ascii=False))

    # Camera trajectory plot with forward axes.
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(camera_centers[:, 0], camera_centers[:, 1], camera_centers[:, 2], marker="o", label="camera path")
    step = max(1, len(camera_centers) // 20)
    scale = max(camera_extent * 0.1, bbox_diag * 0.02, 1e-3)
    ax.quiver(
        camera_centers[::step, 0], camera_centers[::step, 1], camera_centers[::step, 2],
        forward_axes[::step, 0], forward_axes[::step, 1], forward_axes[::step, 2],
        length=scale,
        normalize=True,
        color="tab:red",
        label="OpenCV +Z forward",
    )
    ax.set_title("DA3 camera trajectory and forward axes")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.legend()
    plt.tight_layout()
    trajectory_plot_path = quality_dir / "camera_trajectory.png"
    plt.savefig(trajectory_plot_path, dpi=160)
    plt.show()

    # Point cloud bbox scatter preview.
    preview_points = points_finite
    preview_colors = colors[finite_mask] if len(colors) == len(points) else None
    if len(preview_points) > 30000:
        idx = np.random.default_rng(42).choice(len(preview_points), size=30000, replace=False)
        preview_points = preview_points[idx]
        preview_colors = preview_colors[idx] if preview_colors is not None else None
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    if preview_colors is not None:
        ax.scatter(preview_points[:, 0], preview_points[:, 1], preview_points[:, 2], c=np.clip(preview_colors, 0, 1), s=0.4)
    else:
        ax.scatter(preview_points[:, 0], preview_points[:, 1], preview_points[:, 2], s=0.4)
    ax.set_title("Initial point cloud bbox sanity preview")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    plt.tight_layout()
    pointcloud_plot_path = quality_dir / "init_pointcloud_bbox_preview.png"
    plt.savefig(pointcloud_plot_path, dpi=160)
    plt.show()

    if auto_normalize_scene:
        center = points_finite.mean(axis=0)
        scale_value = max(float(np.percentile(np.linalg.norm(points_finite - center, axis=1), 95)), 1e-6)
        normalized_points = (points - center) / scale_value
        normalized_pcd = o3d.geometry.PointCloud()
        normalized_pcd.points = o3d.utility.Vector3dVector(normalized_points.astype(np.float64))
        if len(colors) == len(points):
            normalized_pcd.colors = o3d.utility.Vector3dVector(colors.astype(np.float64))
        normalized_path = quality_dir / "init_points_normalized_preview.ply"
        o3d.io.write_point_cloud(str(normalized_path), normalized_pcd, write_ascii=False)
        norm_info = {
            "normalized_preview_ply": str(normalized_path),
            "center": center.tolist(),
            "scale": scale_value,
            "note": "This normalized PLY is a preview only. Primary dataset is not mutated automatically.",
        }
        (quality_dir / "normalization_preview_transform.json").write_text(json.dumps(norm_info, indent=2), encoding="utf-8")
        print("Wrote normalization preview. Primary training dataset was not changed:")
        print(json.dumps(norm_info, indent=2))

    if warnings:
        print("Quality warnings:")
        for w in warnings:
            print(f"  - {w}")
    else:
        print("No major camera/scene quality warnings detected.")

    return report


quality_report = validate_da3_camera_and_scene_quality(config)

# 14. gsplat Repository Setup

Clones and inspects the official gsplat repository. The notebook does not invent CLI arguments for `simple_trainer.py`.

In [ ]:
# ============================================================
# 14. gsplat repository setup
# ============================================================

import re
import json
import subprocess
import sys
from pathlib import Path
from types import SimpleNamespace


def recover_config_for_gsplat_if_needed():
    if "config" in globals():
        return globals()["config"]

    drive_root = Path("/content/drive/MyDrive/da3_3dgs_colab")
    if not drive_root.exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as exc:
            print(f"Drive mount attempt skipped/failed: {exc!r}")

    candidates = []
    for root in [drive_root, Path("/content/da3_3dgs_colab")]:
        outputs = root / "outputs"
        if outputs.exists():
            candidates.extend(outputs.glob("*/config.json"))

    preferred = drive_root / "outputs" / "smoke_a100_001" / "config.json"
    if preferred.exists():
        config_path = preferred
    elif candidates:
        config_path = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    else:
        raise NameError("config is not defined and no saved config.json was found. Run cells 4-5 once, then rerun cell 14.")

    data = json.loads(config_path.read_text(encoding="utf-8"))
    cfg = SimpleNamespace(**data)
    cfg.gsplat_repo_revision = getattr(cfg, "gsplat_repo_revision", "")
    cfg.repo_update_policy = getattr(cfg, "repo_update_policy", "reuse_existing")
    cfg.install_gsplat_from_pip = getattr(cfg, "install_gsplat_from_pip", False)
    cfg.gsplat_repo_url = getattr(cfg, "gsplat_repo_url", "https://github.com/nerfstudio-project/gsplat")
    cfg.gsplat_repo_dir = getattr(cfg, "gsplat_repo_dir", "/content/gsplat")
    cfg.output_dir = getattr(cfg, "output_dir", str(config_path.parent))
    cfg.result_ply_path = getattr(cfg, "result_ply_path", str(config_path.parent / "result" / "gaussians.ply"))
    print(f"Recovered config from: {config_path}")
    return cfg


config = recover_config_for_gsplat_if_needed()


if "run_shell" not in globals():
    def run_shell(command, cwd=None, check=True):
        print(f"\n$ {command}")
        result = subprocess.run(
            command,
            cwd=str(cwd) if cwd is not None else None,
            shell=True,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        print(result.stdout)
        if check and result.returncode != 0:
            raise RuntimeError(f"Command failed with exit code {result.returncode}: {command}")
        return result

EXPECTED_GSPLAT_REPO_URL = "https://github.com/nerfstudio-project/gsplat"
GSPLAT_REPO_DIR = Path(config.gsplat_repo_dir)


def setup_gsplat_repository(config):
    if config.gsplat_repo_url != EXPECTED_GSPLAT_REPO_URL:
        raise ValueError(f"gsplat repo must be {EXPECTED_GSPLAT_REPO_URL}")
    if config.install_gsplat_from_pip:
        run_shell(f"{sys.executable} -m pip install gsplat")
    if GSPLAT_REPO_DIR.exists():
        print(f"gsplat repo exists: {GSPLAT_REPO_DIR}")
        if (GSPLAT_REPO_DIR / ".git").exists() and config.repo_update_policy == "pull_latest":
            run_shell("git pull", cwd=GSPLAT_REPO_DIR, check=False)
        else:
            print("Repo update policy is reuse_existing; skipping git pull for reproducibility.")
    else:
        run_shell(f"git clone {config.gsplat_repo_url} {GSPLAT_REPO_DIR}", cwd="/content")

    if config.gsplat_repo_revision and (GSPLAT_REPO_DIR / ".git").exists():
        run_shell(f"git checkout {config.gsplat_repo_revision}", cwd=GSPLAT_REPO_DIR)

    glm_header = GSPLAT_REPO_DIR / "gsplat" / "cuda" / "csrc" / "third_party" / "glm" / "glm" / "gtc" / "type_ptr.hpp"
    if (GSPLAT_REPO_DIR / ".git").exists() and not glm_header.exists():
        print(f"Missing gsplat third-party GLM header: {glm_header}")
        print("Initializing gsplat git submodules recursively.")
        run_shell("git submodule update --init --recursive", cwd=GSPLAT_REPO_DIR)
    if not glm_header.exists():
        raise FileNotFoundError(
            f"Missing required gsplat GLM header after submodule init: {glm_header}. "
            "The CUDA extension cannot build without gsplat/cuda/csrc/third_party/glm."
        )

    commit_result = run_shell("git rev-parse HEAD", cwd=GSPLAT_REPO_DIR, check=False) if (GSPLAT_REPO_DIR / ".git").exists() else None
    gsplat_commit = commit_result.stdout.strip() if commit_result and commit_result.returncode == 0 else "unknown"

    examples_dir = GSPLAT_REPO_DIR / "examples"
    simple = examples_dir / "simple_trainer.py"
    structure = {
        "repo_dir": str(GSPLAT_REPO_DIR),
        "gsplat_dir_exists": (GSPLAT_REPO_DIR / "gsplat").exists(),
        "examples_dir_exists": examples_dir.exists(),
        "simple_trainer_exists": simple.exists(),
        "pyproject_toml_exists": (GSPLAT_REPO_DIR / "pyproject.toml").exists(),
        "setup_py_exists": (GSPLAT_REPO_DIR / "setup.py").exists(),
        "glm_header_exists": glm_header.exists(),
        "git_commit": gsplat_commit,
        "repo_update_policy": config.repo_update_policy,
        "requested_revision": config.gsplat_repo_revision,
    }
    print(json.dumps(structure, indent=2))
    requirements_path = examples_dir / "requirements.txt"
    if requirements_path.exists():
        filtered_requirements = Path(config.output_dir) / "gsplat_examples_requirements.filtered.txt"
        skipped_requirements = []
        kept_lines = []
        for line in requirements_path.read_text(encoding="utf-8").splitlines():
            if "ppisp" in line.lower():
                skipped_requirements.append(line)
                continue
            kept_lines.append(line)
        filtered_requirements.write_text("\n".join(kept_lines) + "\n", encoding="utf-8")
        print("Installing gsplat example requirements with optional ppisp skipped.")
        print("Skipped requirement lines:", skipped_requirements)
        run_shell(f"{sys.executable} -m pip install -r {filtered_requirements}", cwd=GSPLAT_REPO_DIR)
    install_mode = "pypi_requested" if config.install_gsplat_from_pip else "unknown"
    if not config.install_gsplat_from_pip and ((GSPLAT_REPO_DIR / "pyproject.toml").exists() or (GSPLAT_REPO_DIR / "setup.py").exists()):
        editable_result = run_shell(f"{sys.executable} -m pip install -e .", cwd=GSPLAT_REPO_DIR, check=False)
        if editable_result.returncode == 0:
            install_mode = "editable_source"
        else:
            print("Editable source install failed. Falling back to PyPI gsplat, which official gsplat recommends as the easiest install path.")
            print("Training will run from /content so the local repo does not shadow the PyPI package.")
            run_shell(f"{sys.executable} -m pip install gsplat", cwd="/content")
            install_mode = "pypi_fallback"
    run_shell(f"{sys.executable} -c \"import gsplat; print('gsplat import:', gsplat.__file__)\"", cwd="/content")
    if not simple.exists():
        raise FileNotFoundError(f"Missing {simple}")
    source = simple.read_text(encoding="utf-8")
    flags = sorted(set(re.findall(r'''["'](--[A-Za-z0-9_-]+)["']''', source)))
    inspection = {
        "simple_trainer_path": str(simple),
        "mentions_colmap": "colmap" in source.lower(),
        "mentions_transforms_json": "transforms.json" in source,
        "mentions_data_dir": "data_dir" in source,
        "mentions_result_dir": "result_dir" in source,
        "mentions_max_steps": "max_steps" in source,
        "mentions_iterations": "iterations" in source,
        "mentions_save_ply": "save_ply" in source,
        "cli_flags_found_in_source": flags,
        "install_mode": install_mode,
        "note": "Do not invent CLI flags. Use only verified args from the current simple_trainer.py.",
    }
    print(json.dumps(inspection, indent=2))
    help_result = run_shell(f"{sys.executable} {simple} --help", cwd=GSPLAT_REPO_DIR, check=False)
    inspection["help_returncode"] = help_result.returncode
    inspection["help_output_tail"] = help_result.stdout[-4000:]
    (Path(config.output_dir) / "gsplat_repository_setup.json").write_text(json.dumps({"structure": structure, "inspection": inspection}, indent=2), encoding="utf-8")
    return inspection


gsplat_trainer_inspection = setup_gsplat_repository(config)

# 15. DA3-to-gsplat Dataset Conversion

Creates an intermediate dataset:

- `gsplat_dataset/images/`
- `gsplat_dataset/transforms.json`
- `gsplat_dataset/points3D.ply`

This `transforms.json` format is not guaranteed to be directly compatible with every gsplat trainer. If `simple_trainer.py` requires COLMAP, implement DA3-to-COLMAP conversion or a verified custom loader.

In [ ]:
# ============================================================
# 15. DA3-to-gsplat dataset conversion
# ============================================================

def make_c2w(extrinsic, extrinsics_type):
    mat = np.eye(4, dtype=np.float32)
    mat[:3, :4] = np.asarray(extrinsic, dtype=np.float32)
    if extrinsics_type == "w2c":
        return np.linalg.inv(mat).astype(np.float32)
    if extrinsics_type == "c2w":
        return mat.astype(np.float32)
    raise ValueError(f"Unsupported extrinsics_type={extrinsics_type}")


def convert_da3_to_gsplat_dataset(config):
    dataset_dir = Path(config.gsplat_dataset_dir)
    images_dir = dataset_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    transforms_path = dataset_dir / "transforms.json"
    points3d_path = dataset_dir / "points3D.ply"
    if config.resume and transforms_path.exists() and points3d_path.exists() and not config.overwrite:
        print("Using existing gsplat intermediate dataset.")
        return transforms_path, points3d_path
    frames = json.loads((Path(config.output_dir) / "frames.json").read_text(encoding="utf-8"))
    cameras = json.loads((Path(config.da3_output_dir) / "cameras.json").read_text(encoding="utf-8"))
    camera_frames = cameras["frames"]
    if len(frames) != len(camera_frames):
        raise ValueError("frames.json and cameras.json counts differ.")
    init_points = Path(config.pointcloud_dir) / "init_points.ply"
    if not init_points.exists():
        raise FileNotFoundError(f"Missing {init_points}")
    K0 = np.asarray(camera_frames[0]["intrinsic"], dtype=np.float32)
    transform_frames = []
    for idx, item in enumerate(camera_frames):
        src = Path(item["image_path"])
        dst = images_dir / f"frame_{idx+1:06d}{src.suffix.lower()}"
        if not dst.exists() or config.overwrite:
            shutil.copy2(src, dst)
        extrinsics_type = item.get("extrinsics_type", cameras.get("extrinsics_type", config.extrinsics_type))
        c2w = make_c2w(item["extrinsic"], extrinsics_type)
        transform_frames.append({"file_path": f"images/{dst.name}", "transform_matrix": c2w.tolist()})
    shutil.copy2(init_points, points3d_path)
    transforms = {
        "fl_x": float(K0[0, 0]),
        "fl_y": float(K0[1, 1]),
        "cx": float(K0[0, 2]),
        "cy": float(K0[1, 2]),
        "w": int(cameras.get("width", frames[0]["width"])),
        "h": int(cameras.get("height", frames[0]["height"])),
        "camera_model": "OPENCV",
        "frames": transform_frames,
        "metadata": {
            "warning": "This transforms.json may require a custom gsplat loader. If simple_trainer requires COLMAP, TODO: implement DA3-to-COLMAP conversion.",
            "points3D_ply": str(points3d_path),
        },
    }
    transforms_path.write_text(json.dumps(transforms, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved {transforms_path}")
    print(f"Saved {points3d_path}")
    print("TODO if needed: adapt simple_trainer.py dataset loader or convert to COLMAP format.")
    return transforms_path, points3d_path


transforms_json_path, points3d_ply_path = convert_da3_to_gsplat_dataset(config)

# 15.2 COLMAP Text Dataset Fallback

This fallback writes a COLMAP-like text dataset under:

```text
outputs/{job_id}/work/gsplat_dataset_colmap/
  images/
  sparse/0/cameras.txt
  sparse/0/images.txt
  sparse/0/points3D.txt
```

It is not the primary route. The primary route remains the DA3 custom loader and `examples/da3_trainer.py`.

Use this fallback only if the custom DA3 trainer path breaks and you want to try gsplat's native COLMAP parser. The conversion writes a minimal COLMAP text model with PINHOLE cameras, DA3 camera poses, and initialization points. 2D-3D observations are intentionally left empty because gsplat initialization mainly needs camera poses, intrinsics, images, and 3D points/colors.

In [ ]:
# ============================================================
# 15.2 COLMAP text dataset fallback
# ============================================================

from pathlib import Path
import json
import shutil
import numpy as np
from plyfile import PlyData


def _rotmat_to_qvec_colmap(R):
    """Convert rotation matrix to COLMAP qvec [qw, qx, qy, qz]."""
    R = np.asarray(R, dtype=np.float64)
    K = np.array([
        [R[0, 0] - R[1, 1] - R[2, 2], 0, 0, 0],
        [R[1, 0] + R[0, 1], R[1, 1] - R[0, 0] - R[2, 2], 0, 0],
        [R[2, 0] + R[0, 2], R[2, 1] + R[1, 2], R[2, 2] - R[0, 0] - R[1, 1], 0],
        [R[1, 2] - R[2, 1], R[2, 0] - R[0, 2], R[0, 1] - R[1, 0], R[0, 0] + R[1, 1] + R[2, 2]],
    ]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    q = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if q[0] < 0:
        q *= -1
    return q / np.linalg.norm(q)


def _read_init_points_for_colmap(points_ply_path, max_points=None):
    ply = PlyData.read(str(points_ply_path))
    vertex = ply["vertex"].data
    names = set(vertex.dtype.names)
    points = np.stack([
        np.asarray(vertex["x"], dtype=np.float64),
        np.asarray(vertex["y"], dtype=np.float64),
        np.asarray(vertex["z"], dtype=np.float64),
    ], axis=1)
    if {"red", "green", "blue"}.issubset(names):
        colors = np.stack([
            np.asarray(vertex["red"], dtype=np.uint8),
            np.asarray(vertex["green"], dtype=np.uint8),
            np.asarray(vertex["blue"], dtype=np.uint8),
        ], axis=1)
    else:
        colors = np.full((len(points), 3), 127, dtype=np.uint8)
    finite = np.isfinite(points).all(axis=1)
    points = points[finite]
    colors = colors[finite]
    if max_points and len(points) > max_points:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(points), size=int(max_points), replace=False)
        points, colors = points[idx], colors[idx]
    return points, colors


def _extrinsic_to_w2c_4x4(extrinsic, extrinsics_type):
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :4] = np.asarray(extrinsic, dtype=np.float64)
    if extrinsics_type == "w2c":
        return mat
    if extrinsics_type == "c2w":
        return np.linalg.inv(mat)
    raise ValueError(f"Unsupported extrinsics_type={extrinsics_type}")


def write_colmap_text_fallback_dataset(config):
    source_dataset_dir = Path(config.gsplat_dataset_dir)
    fallback_dir = Path(config.output_dir) / "work" / "gsplat_dataset_colmap"
    images_dir = fallback_dir / "images"
    sparse_dir = fallback_dir / "sparse" / "0"
    images_dir.mkdir(parents=True, exist_ok=True)
    sparse_dir.mkdir(parents=True, exist_ok=True)

    cameras_json_path = Path(config.da3_output_dir) / "cameras.json"
    frames_json_path = Path(config.output_dir) / "frames.json"
    init_points_path = Path(config.pointcloud_dir) / "init_points.ply"

    if not cameras_json_path.exists():
        raise FileNotFoundError(cameras_json_path)
    if not frames_json_path.exists():
        raise FileNotFoundError(frames_json_path)
    if not init_points_path.exists():
        raise FileNotFoundError(init_points_path)

    cameras_json = json.loads(cameras_json_path.read_text(encoding="utf-8"))
    camera_frames = cameras_json.get("frames", [])
    if not camera_frames:
        raise ValueError("No DA3 camera frames found for COLMAP fallback conversion.")

    # Copy images and build one PINHOLE camera per image. This avoids assuming fixed intrinsics.
    cameras_lines = [
        "# Camera list with one line of data per camera:\n",
        "#   CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n",
        "# Number of cameras: {}\n".format(len(camera_frames)),
    ]
    images_lines = [
        "# Image list with two lines of data per image:\n",
        "#   IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n",
        "#   POINTS2D[] as (X, Y, POINT3D_ID)\n",
        "# Number of images: {}, mean observations per image: 0\n".format(len(camera_frames)),
    ]

    for idx, item in enumerate(camera_frames, start=1):
        src_image = Path(item["image_path"])
        dst_name = f"frame_{idx:06d}{src_image.suffix.lower()}"
        dst_image = images_dir / dst_name
        if not dst_image.exists() or config.overwrite:
            shutil.copy2(src_image, dst_image)

        K = np.asarray(item["intrinsic"], dtype=np.float64)
        width = int(cameras_json.get("width", item.get("width", 0)))
        height = int(cameras_json.get("height", item.get("height", 0)))
        if width <= 0 or height <= 0:
            from PIL import Image
            with Image.open(dst_image) as im:
                width, height = im.size

        fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
        camera_id = idx
        cameras_lines.append(f"{camera_id} PINHOLE {width} {height} {fx:.12g} {fy:.12g} {cx:.12g} {cy:.12g}\n")

        extrinsics_type = item.get("extrinsics_type", cameras_json.get("extrinsics_type", config.extrinsics_type))
        w2c = _extrinsic_to_w2c_4x4(item["extrinsic"], extrinsics_type)
        qvec = _rotmat_to_qvec_colmap(w2c[:3, :3])
        tvec = w2c[:3, 3]
        images_lines.append(
            f"{idx} {qvec[0]:.17g} {qvec[1]:.17g} {qvec[2]:.17g} {qvec[3]:.17g} "
            f"{tvec[0]:.17g} {tvec[1]:.17g} {tvec[2]:.17g} {camera_id} {dst_name}\n"
        )
        images_lines.append("\n")

    # Write sparse points. TRACK[] is intentionally empty.
    points, colors = _read_init_points_for_colmap(init_points_path, max_points=config.max_points)
    points_lines = [
        "# 3D point list with one line of data per point:\n",
        "#   POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[] as (IMAGE_ID, POINT2D_IDX)\n",
        "# Number of points: {}, mean track length: 0\n".format(len(points)),
    ]
    for idx, (point, color) in enumerate(zip(points, colors), start=1):
        r, g, b = [int(x) for x in color]
        points_lines.append(
            f"{idx} {point[0]:.17g} {point[1]:.17g} {point[2]:.17g} {r} {g} {b} 0\n"
        )

    (sparse_dir / "cameras.txt").write_text("".join(cameras_lines), encoding="utf-8")
    (sparse_dir / "images.txt").write_text("".join(images_lines), encoding="utf-8")
    (sparse_dir / "points3D.txt").write_text("".join(points_lines), encoding="utf-8")

    manifest = {
        "type": "colmap_text_fallback",
        "primary_route": "DA3 custom loader + examples/da3_trainer.py",
        "fallback_dir": str(fallback_dir),
        "images_dir": str(images_dir),
        "sparse_dir": str(sparse_dir),
        "camera_count": len(camera_frames),
        "point_count": int(len(points)),
        "warning": (
            "This is a minimal COLMAP text model. It has camera poses, intrinsics, images, and points, "
            "but no 2D-3D tracks. Use only as fallback if the DA3 custom trainer path fails."
        ),
        "usage_hint": (
            "To try this fallback, set config.gsplat_command_template to a verified simple_trainer command "
            "that uses data_dir=<fallback_dir>. Do not invent CLI flags; inspect simple_trainer.py help first."
        ),
    }
    (fallback_dir / "colmap_fallback_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    print("COLMAP text fallback dataset written:")
    print(f"  {fallback_dir}")
    print(f"  cameras.txt: {sparse_dir / 'cameras.txt'}")
    print(f"  images.txt: {sparse_dir / 'images.txt'}")
    print(f"  points3D.txt: {sparse_dir / 'points3D.txt'}")
    print("Primary route remains DA3 custom loader + da3_trainer.py.")
    return fallback_dir


colmap_fallback_dir = write_colmap_text_fallback_dataset(config)

# 15.1 Verified DA3-to-gsplat Trainer Bridge

The current gsplat `examples/simple_trainer.py` expects a COLMAP-like `Parser` and `Dataset` interface.

Observed interface:

- `Parser`
  - `image_names`
  - `image_paths`
  - `camtoworlds`
  - `camera_ids`
  - `camera_indices`
  - `Ks_dict`
  - `params_dict`
  - `imsize_dict`
  - `mask_dict`
  - `points`
  - `points_rgb`
  - `scene_scale`
  - `num_cameras`
- `Dataset.__getitem__`
  - `image`
  - `K`
  - `camtoworld`
  - `image_id`
  - `camera_idx`
  - optional `mask`

This bridge writes:

```text
/content/gsplat/examples/datasets/da3.py
/content/gsplat/examples/da3_train_entry.py
```

The entry script reuses gsplat's official `simple_trainer.py` training `Runner`, initialization, rasterization, densification, and `gsplat.export_splats(...)` export flow, but injects a DA3-compatible Parser/Dataset.

In [ ]:
# ============================================================
# 15.1 Verified DA3-to-gsplat trainer bridge
# ============================================================

from pathlib import Path
import textwrap


def write_da3_gsplat_bridge(config):
    repo_dir = Path(config.gsplat_repo_dir)
    examples_dir = repo_dir / "examples"
    datasets_dir = examples_dir / "datasets"
    simple_trainer_path = examples_dir / "simple_trainer.py"

    if not simple_trainer_path.exists():
        raise FileNotFoundError(f"Missing gsplat simple_trainer.py: {simple_trainer_path}")

    datasets_dir.mkdir(parents=True, exist_ok=True)

    da3_dataset_path = datasets_dir / "da3.py"
    da3_entry_path = examples_dir / "da3_train_entry.py"
    da3_trainer_path = examples_dir / "da3_trainer.py"
    datasets_init_path = datasets_dir / "__init__.py"

    da3_dataset_code = r"""
import json
from pathlib import Path
from typing import Any, Dict, Optional

import cv2
import imageio.v2 as imageio
import numpy as np
import torch
from plyfile import PlyData


def _load_points_ply(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing initialization point cloud: {path}")

    ply = PlyData.read(str(path))
    vertex = ply["vertex"].data
    names = vertex.dtype.names

    points = np.stack(
        [
            np.asarray(vertex["x"], dtype=np.float32),
            np.asarray(vertex["y"], dtype=np.float32),
            np.asarray(vertex["z"], dtype=np.float32),
        ],
        axis=1,
    )

    if {"red", "green", "blue"}.issubset(set(names)):
        colors = np.stack(
            [
                np.asarray(vertex["red"], dtype=np.uint8),
                np.asarray(vertex["green"], dtype=np.uint8),
                np.asarray(vertex["blue"], dtype=np.uint8),
            ],
            axis=1,
        )
    else:
        colors = np.full((points.shape[0], 3), 127, dtype=np.uint8)

    finite = np.isfinite(points).all(axis=1)
    points = points[finite]
    colors = colors[finite]

    if points.shape[0] == 0:
        raise ValueError(f"No valid points in {path}")

    return points.astype(np.float32), colors.astype(np.uint8)


class DA3Parser:
    # DA3 transforms.json parser matching gsplat examples/simple_trainer.py needs.

    def __init__(
        self,
        data_dir: str,
        factor: int = 1,
        normalize: bool = False,
        test_every: int = 8,
        load_exposure: bool = False,
    ):
        self.data_dir = str(data_dir)
        self.factor = int(factor)
        self.normalize = bool(normalize)
        self.test_every = max(1, int(test_every))
        self.load_exposure = bool(load_exposure)

        data_dir = Path(data_dir)
        transforms_path = data_dir / "transforms.json"
        points_path = data_dir / "points3D.ply"

        if not transforms_path.exists():
            raise FileNotFoundError(f"Missing transforms.json: {transforms_path}")

        with open(transforms_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        frames = meta.get("frames", [])
        if not frames:
            raise ValueError(f"No frames found in {transforms_path}")

        image_names = []
        image_paths = []
        camtoworlds = []
        camera_ids = []
        Ks_dict = {}
        params_dict = {}
        imsize_dict = {}
        mask_dict = {}

        default_K = np.array(
            [
                [float(meta["fl_x"]), 0.0, float(meta["cx"])],
                [0.0, float(meta["fl_y"]), float(meta["cy"])],
                [0.0, 0.0, 1.0],
            ],
            dtype=np.float32,
        )
        default_w = int(meta["w"])
        default_h = int(meta["h"])

        for idx, frame in enumerate(frames):
            rel = frame["file_path"]
            image_path = data_dir / rel
            if not image_path.exists():
                raise FileNotFoundError(f"Missing image referenced by transforms.json: {image_path}")

            c2w = np.asarray(frame["transform_matrix"], dtype=np.float32)
            if c2w.shape != (4, 4):
                raise ValueError(f"Expected transform_matrix [4,4] at frame {idx}, got {c2w.shape}")

            if "intrinsic" in frame:
                K = np.asarray(frame["intrinsic"], dtype=np.float32)
            else:
                K = default_K.copy()

            if K.shape != (3, 3):
                raise ValueError(f"Expected K [3,3] at frame {idx}, got {K.shape}")

            camera_id = idx
            image_names.append(Path(rel).name)
            image_paths.append(str(image_path))
            camtoworlds.append(c2w)
            camera_ids.append(camera_id)
            Ks_dict[camera_id] = K
            params_dict[camera_id] = np.array([], dtype=np.float32)

            # Trust actual image size over metadata if available.
            image = imageio.imread(str(image_path))[..., :3]
            h, w = image.shape[:2]
            imsize_dict[camera_id] = (int(w), int(h))
            mask_dict[camera_id] = None

        points, points_rgb = _load_points_ply(points_path)

        self.image_names = image_names
        self.image_paths = image_paths
        self.camtoworlds = np.stack(camtoworlds, axis=0).astype(np.float32)
        self.camera_ids = camera_ids
        self.Ks_dict = Ks_dict
        self.params_dict = params_dict
        self.imsize_dict = imsize_dict
        self.mask_dict = mask_dict
        self.points = points
        self.points_err = np.zeros((points.shape[0],), dtype=np.float32)
        self.points_rgb = points_rgb
        self.point_indices = {name: np.array([], dtype=np.int64) for name in image_names}
        self.transform = np.eye(4, dtype=np.float32)
        self.extconf = {"spiral_radius_scale": 1.0}
        self.bounds = np.array([0.01, 1e10], dtype=np.float32)
        self.exposure_values = [None] * len(image_paths)

        self.camera_id_to_idx = {cid: idx for idx, cid in enumerate(sorted(set(camera_ids)))}
        self.camera_indices = [self.camera_id_to_idx[cid] for cid in camera_ids]
        self.num_cameras = len(self.camera_id_to_idx)

        camera_locations = self.camtoworlds[:, :3, 3]
        center = camera_locations.mean(axis=0)
        camera_extent = np.linalg.norm(camera_locations - center, axis=1).max()
        point_extent = np.linalg.norm(points - points.mean(axis=0), axis=1).max()
        self.scene_scale = float(max(camera_extent, point_extent * 0.1, 1e-3))

        if normalize:
            print("[DA3Parser] normalize=True was requested, but DA3 bridge keeps DA3 world scale unchanged.")


class DA3Dataset:
    # Dataset output schema matched to gsplat examples/simple_trainer.py.

    def __init__(
        self,
        parser: DA3Parser,
        split: str = "train",
        patch_size: Optional[int] = None,
        load_depths: bool = False,
    ):
        self.parser = parser
        self.split = split
        self.patch_size = patch_size
        self.load_depths = load_depths

        indices = np.arange(len(self.parser.image_names))
        if len(indices) == 1:
            self.indices = indices
        elif split == "train":
            self.indices = indices[indices % self.parser.test_every != 0]
            if len(self.indices) == 0:
                self.indices = indices
        else:
            self.indices = indices[indices % self.parser.test_every == 0]
            if len(self.indices) == 0:
                self.indices = indices[:1]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item: int) -> Dict[str, Any]:
        index = int(self.indices[item])
        image = imageio.imread(self.parser.image_paths[index])[..., :3]
        camera_id = self.parser.camera_ids[index]
        K = self.parser.Ks_dict[camera_id].copy()
        camtoworld = self.parser.camtoworlds[index].copy()
        mask = self.parser.mask_dict[camera_id]

        if self.patch_size is not None:
            h, w = image.shape[:2]
            x = np.random.randint(0, max(w - self.patch_size, 1))
            y = np.random.randint(0, max(h - self.patch_size, 1))
            image = image[y : y + self.patch_size, x : x + self.patch_size]
            K[0, 2] -= x
            K[1, 2] -= y
            if mask is not None:
                mask = mask[y : y + self.patch_size, x : x + self.patch_size]

        data = {
            "K": torch.from_numpy(K).float(),
            "camtoworld": torch.from_numpy(camtoworld).float(),
            "image": torch.from_numpy(image).float(),
            "image_id": torch.tensor(item, dtype=torch.long),
            "camera_idx": torch.tensor(self.parser.camera_indices[index], dtype=torch.long),
        }

        if mask is not None:
            data["mask"] = torch.from_numpy(mask).bool()

        if self.load_depths:
            raise NotImplementedError(
                "DA3 depth_loss samples are not wired in this bridge. "
                "Run with depth_loss=False, or extend DA3Dataset to return points/depths."
            )

        return data
"""

    da3_entry_code = r"""
import argparse
from pathlib import Path

import torch

from datasets.da3 import DA3Dataset, DA3Parser
from gsplat import export_splats
import simple_trainer


FINAL_GAUSSIAN_NOTE = "This is a 3DGS Gaussian PLY, not a triangle mesh PLY."


def export_final_gaussian_ply(runner, save_to: Path) -> None:
    # Explicitly export the trained Gaussian primitives with gsplat.export_splats.
    save_to = Path(save_to)
    save_to.parent.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        if runner.cfg.app_opt:
            rgb = runner.app_module(
                features=runner.splats["features"],
                embed_ids=None,
                dirs=torch.zeros_like(runner.splats["means"][None, :, :]),
                sh_degree=runner.cfg.sh_degree,
            )
            rgb = rgb + runner.splats["colors"]
            rgb = torch.sigmoid(rgb).squeeze(0).unsqueeze(1)
            sh0 = simple_trainer.rgb_to_sh(rgb)
            shN = torch.empty([sh0.shape[0], 0, 3], device=sh0.device)
        else:
            sh0 = runner.splats["sh0"]
            shN = runner.splats["shN"]

        export_splats(
            means=runner.splats["means"],
            scales=runner.splats["scales"],
            quats=runner.splats["quats"],
            opacities=runner.splats["opacities"],
            sh0=sh0,
            shN=shN,
            format="ply",
            save_to=str(save_to),
        )

    if not save_to.exists():
        raise RuntimeError(f"gsplat.export_splats did not create {save_to}")

    print(f"Explicit gsplat Gaussian PLY export completed: {save_to}")
    print(FINAL_GAUSSIAN_NOTE)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset-dir", required=True)
    parser.add_argument("--train-dir", required=True)
    parser.add_argument("--iterations", type=int, required=True)
    parser.add_argument("--points-ply", required=True)
    parser.add_argument("--test-every", type=int, default=8)
    parser.add_argument("--data-factor", type=int, default=1)
    parser.add_argument(
        "--final-ply",
        default=None,
        help="Explicit gsplat.export_splats output path. This notebook passes outputs/{job_id}/result/gaussians.ply.",
    )
    args = parser.parse_args()

    dataset_dir = Path(args.dataset_dir)
    train_dir = Path(args.train_dir)
    points_ply = Path(args.points_ply)
    final_ply = Path(args.final_ply) if args.final_ply else train_dir / "ply" / "gaussians.ply"

    if not (dataset_dir / "transforms.json").exists():
        raise FileNotFoundError(f"Missing transforms.json: {dataset_dir / 'transforms.json'}")
    if not points_ply.exists():
        raise FileNotFoundError(f"Missing points ply: {points_ply}")

    # Reuse official simple_trainer Runner while replacing the COLMAP Parser/Dataset
    # globals with DA3-compatible implementations. This avoids relying on guessed
    # simple_trainer CLI flags or modifying the upstream trainer source.
    simple_trainer.Parser = DA3Parser
    simple_trainer.Dataset = DA3Dataset

    cfg = simple_trainer.Config()
    cfg.disable_viewer = True
    cfg.disable_video = True
    cfg.data_type = "colmap"  # enters the Parser/Dataset branch, monkey-patched above
    cfg.data_dir = str(dataset_dir)
    cfg.data_factor = int(args.data_factor)
    cfg.result_dir = str(train_dir)
    cfg.test_every = max(1, int(args.test_every))
    cfg.batch_size = 1
    cfg.max_steps = int(args.iterations)
    cfg.eval_steps = []
    cfg.save_steps = [int(args.iterations)]

    # Disable simple_trainer's periodic PLY write to avoid ambiguity with generic
    # point_cloud_*.ply names. The final PLY below is explicitly written via
    # gsplat.export_splats(..., format="ply", save_to=<train-dir>/ply/gaussians.ply).
    cfg.save_ply = False
    cfg.ply_steps = []

    cfg.init_type = "sfm"
    cfg.normalize_world_space = False
    cfg.load_exposure = False
    cfg.depth_loss = False

    runner = simple_trainer.Runner(local_rank=0, world_rank=0, world_size=1, cfg=cfg)
    runner.train()
    runner.export_ppisp_reports()

    export_final_gaussian_ply(runner, final_ply)


if __name__ == "__main__":
    main()

"""

    da3_dataset_path.write_text(textwrap.dedent(da3_dataset_code).lstrip(), encoding="utf-8")
    da3_entry_path.write_text(textwrap.dedent(da3_entry_code).lstrip(), encoding="utf-8")
    da3_trainer_path.write_text(textwrap.dedent(da3_entry_code).replace("da3_train_entry.py", "da3_trainer.py").lstrip(), encoding="utf-8")
    if not datasets_init_path.exists():
        datasets_init_path.write_text("", encoding="utf-8")

    # Use the verified bridge entry instead of a guessed simple_trainer CLI.
    config.gsplat_command_template = (
        "python {repo_dir}/examples/da3_trainer.py "
        "--dataset-dir {dataset_dir} "
        "--train-dir {train_dir} "
        "--iterations {iterations} "
        "--points-ply {points_ply} "
        "--final-ply {result_ply}"
    )

    print("DA3 gsplat bridge written:")
    print(f"  {da3_dataset_path}")
    print(f"  {da3_entry_path}")
    print(f"  {da3_trainer_path}")
    print(f"  {datasets_init_path}")
    print("Updated config.gsplat_command_template:")
    print(config.gsplat_command_template)
    print()
    print("The final training export uses gsplat.export_splats inside simple_trainer.py.")
    print("Exported PLY candidates will be Gaussian PLY files, not triangle mesh PLY files.")

    return da3_dataset_path, da3_entry_path


da3_dataset_bridge_path, da3_train_entry_path = write_da3_gsplat_bridge(config)

# 16. gsplat Training

Builds the training command from `config.gsplat_command_template`.

If `simple_trainer.py` cannot read this dataset format, the cell stops with a TODO instead of inventing unsupported CLI arguments.

In [ ]:
# ============================================================
# 16. gsplat training
# ============================================================

import time
import os


def find_gsplat_ckpt_candidates(train_dir):
    train_dir = Path(train_dir)
    candidates = sorted(train_dir.glob("**/ckpt_*_rank*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates


def gsplat_training_complete(config):
    final_result = Path(config.result_ply_path)
    train_export = Path(getattr(config, "gsplat_final_train_ply", Path(config.gsplat_train_dir) / "ply" / "gaussians.ply"))
    return (final_result.exists() and final_result.stat().st_size > 0) or (train_export.exists() and train_export.stat().st_size > 0)


def write_resume_state(config, stage, state):
    path = Path(config.output_dir) / "resume_state.json"
    if path.exists():
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            data = {}
    else:
        data = {}
    data[stage] = state
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved resume state: {path}")


def find_train_ply_candidates(train_dir):
    train_dir = Path(train_dir)
    patterns = ["**/point_cloud.ply", "**/point_cloud_*.ply", "**/gaussians.ply", "**/*.ply"]
    candidates = []
    for pattern in patterns:
        for path in train_dir.glob(pattern):
            if path.is_file() and path not in candidates:
                candidates.append(path)
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)


def build_gsplat_training_command(config):
    dataset_dir = Path(config.gsplat_dataset_dir)
    train_dir = Path(config.gsplat_train_dir)
    repo_dir = Path(config.gsplat_repo_dir)
    points_ply = dataset_dir / "points3D.ply"
    train_dir.mkdir(parents=True, exist_ok=True)
    return config.gsplat_command_template.format(
        dataset_dir=str(dataset_dir),
        train_dir=str(train_dir),
        iterations=int(config.gsplat_iterations),
        points_ply=str(points_ply),
        repo_dir=str(repo_dir),
        result_ply=str(Path(config.result_ply_path)),
    )


def validate_gsplat_training_route(config, inspection):
    command = config.gsplat_command_template
    uses_simple_trainer = "simple_trainer.py" in command
    if uses_simple_trainer and inspection.get("mentions_colmap") and not inspection.get("mentions_transforms_json"):
        msg = (
            "The current gsplat simple_trainer.py appears COLMAP-oriented and does not directly read transforms.json.\n"
            "TODO: implement DA3-to-COLMAP conversion, add a verified DA3 Dataset/Parser, or use a custom trainer.\n"
            "Do not invent missing gsplat CLI args."
        )
        if config.dry_run:
            print("[DRY RUN WARNING]\n" + msg)
            return False
        raise RuntimeError(msg)
    return True


def run_gsplat_training(config, inspection):
    train_dir = Path(config.gsplat_train_dir)
    repo_dir = Path(config.gsplat_repo_dir)
    train_dir.mkdir(parents=True, exist_ok=True)
    explicit_gaussian_ply = Path(config.result_ply_path)
    if config.resume and not config.overwrite and explicit_gaussian_ply.exists() and explicit_gaussian_ply.stat().st_size > 0:
        print("Resume enabled and final result/gaussians.ply already exists. Skipping gsplat training.")
        print(f"  {explicit_gaussian_ply}")
        candidates = find_train_ply_candidates(train_dir)
        write_resume_state(config, "gsplat_training", {
            "status": "skipped_complete",
            "gaussian_ply": str(explicit_gaussian_ply),
            "ckpt_candidates": [str(p) for p in find_gsplat_ckpt_candidates(train_dir)],
            "note": "Training was skipped because gsplat.export_splats output already exists.",
        })
        return {"dry_run": False, "skipped": True, "ply_candidates": [str(p) for p in candidates]}

    ckpts = find_gsplat_ckpt_candidates(train_dir)
    if config.resume and ckpts and not explicit_gaussian_ply.exists():
        msg = (
            "Found gsplat checkpoint(s) but no explicit gaussians.ply. "
            "Current simple_trainer checkpoint is suitable for evaluation/export fallback, "
            "but not guaranteed for exact optimizer/strategy training resume."
        )
        print("[RESUME NOTICE] " + msg)
        print(f"Latest checkpoint: {ckpts[0]}")
        write_resume_state(config, "gsplat_training", {
            "status": "partial_checkpoint_found",
            "latest_checkpoint": str(ckpts[0]),
            "policy": config.gsplat_resume_policy,
            "note": msg,
        })
        if config.gsplat_resume_policy == "error_if_incomplete":
            raise RuntimeError(msg + " Set gsplat_resume_policy=restart_if_incomplete to restart training, or run final export fallback from checkpoint.")
        print("Policy restart_if_incomplete: training will restart without deleting existing checkpoints.")

    command = build_gsplat_training_command(config)
    (train_dir / "train_command.txt").write_text(command + "\n", encoding="utf-8")
    print("gsplat training command:")
    print(command)
    print("OOM mitigation: reduce gsplat_iterations, max_frames, resize_width/resize_height, or max_points.")
    route_ok = validate_gsplat_training_route(config, inspection)
    if config.dry_run:
        print("config.dry_run=True. Training will not run.")
        return {"dry_run": True, "route_ok": route_ok, "ply_candidates": [str(p) for p in find_train_ply_candidates(train_dir)]}
    log_path = train_dir / "train.log"
    if log_path.exists() and config.resume and not config.overwrite:
        log_path = train_dir / f"train_restart_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    start = time.time()
    recent = []
    training_cwd = repo_dir
    training_env = os.environ.copy()
    py_paths = [str(repo_dir), str(repo_dir / "examples")]
    existing_pythonpath = training_env.get("PYTHONPATH", "")
    training_env["PYTHONPATH"] = ":".join(py_paths + ([existing_pythonpath] if existing_pythonpath else []))
    if inspection.get("install_mode") == "pypi_fallback":
        print("Using source repo on PYTHONPATH so simple_trainer.py and gsplat package come from the same checkout.")
    with open(log_path, "w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=str(training_cwd), env=training_env, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            log.flush()
            recent.append(line.rstrip())
            recent = recent[-25:]
            if any(k in line.lower() for k in ["step", "iter", "loss", "psnr", "save", "error", "cuda", "out of memory"]):
                print(line.rstrip())
        rc = process.wait()
    if rc != 0:
        print("\n".join(recent))
        raise RuntimeError("gsplat training failed. Check train.log and verified CLI args.")
    candidates = find_train_ply_candidates(train_dir)
    (train_dir / "ply_candidates.json").write_text(json.dumps([str(p) for p in candidates], indent=2), encoding="utf-8")
    write_resume_state(config, "gsplat_training", {
        "status": "completed",
        "elapsed_sec": time.time() - start,
        "ply_candidates": [str(p) for p in candidates],
        "explicit_gaussian_ply": str(Path(getattr(config, "gsplat_final_train_ply", train_dir / "ply" / "gaussians.ply"))),
        "note": "Training completed. Explicit Gaussian PLY should be produced by da3_trainer.py via gsplat.export_splats.",
    })
    return {"dry_run": False, "elapsed_sec": time.time() - start, "ply_candidates": [str(p) for p in candidates]}


training_result = run_gsplat_training(config, gsplat_trainer_inspection)

# 17. Final Gaussian PLY Export / Download / Debug Utilities

Copies the latest trained PLY candidate to:

```text
outputs/{job_id}/result/gaussians.ply
```

Important:

```text
gaussians.ply is a 3DGS Gaussian PLY, not a triangle mesh PLY.
```

In [ ]:
# ============================================================
# 17. Final Gaussian PLY export / download / debug utilities
# ============================================================

import zipfile

FINAL_GAUSSIAN_NOTE = "This is a 3DGS Gaussian PLY, not a triangle mesh PLY."


def debug_pipeline_state(config):
    def load_json(path):
        path = Path(path)
        return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
    frames = load_json(Path(config.output_dir) / "frames.json")
    cameras = load_json(Path(config.da3_output_dir) / "cameras.json")
    transforms = load_json(Path(config.gsplat_dataset_dir) / "transforms.json")
    depth_files = sorted((Path(config.da3_output_dir) / "depth").glob("depth_*.npy"))
    conf_files = sorted((Path(config.da3_output_dir) / "confidence").glob("conf_*.npy"))
    result_ply = Path(config.result_ply_path)
    report = {
        "frame_count": len(frames) if isinstance(frames, list) else None,
        "depth_count": len(depth_files),
        "confidence_count": len(conf_files),
        "cameras_json_frame_count": len(cameras.get("frames", [])) if isinstance(cameras, dict) else None,
        "init_points_ply_exists": (Path(config.pointcloud_dir) / "init_points.ply").exists(),
        "transforms_json_frame_count": len(transforms.get("frames", [])) if isinstance(transforms, dict) else None,
        "train_ply_candidates": [str(p) for p in find_train_ply_candidates(config.gsplat_train_dir)],
        "gsplat_checkpoint_candidates": [str(p) for p in (find_gsplat_ckpt_candidates(config.gsplat_train_dir) if "find_gsplat_ckpt_candidates" in globals() else sorted(Path(config.gsplat_train_dir).glob("**/ckpt_*_rank*.pt"), key=lambda p: p.stat().st_mtime, reverse=True))],
        "result_ply_exists": result_ply.exists(),
        "result_ply_size_mb": result_ply.stat().st_size / (1024 * 1024) if result_ply.exists() else None,
        "note": FINAL_GAUSSIAN_NOTE,
    }
    print(json.dumps(report, indent=2))
    (Path(config.output_dir) / "debug_pipeline_state.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


def export_gaussian_ply_from_checkpoint(ckpt_path, save_to):
    import torch
    from gsplat import export_splats

    ckpt_path = Path(ckpt_path)
    save_to = Path(save_to)
    save_to.parent.mkdir(parents=True, exist_ok=True)
    ckpt = torch.load(str(ckpt_path), map_location="cuda" if torch.cuda.is_available() else "cpu")
    splats = ckpt.get("splats")
    if splats is None:
        raise RuntimeError(f"Checkpoint has no splats state_dict: {ckpt_path}")
    required = ["means", "scales", "quats", "opacities", "sh0", "shN"]
    missing = [k for k in required if k not in splats]
    if missing:
        raise RuntimeError(f"Cannot export checkpoint to Gaussian PLY; missing splat keys: {missing}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    export_splats(
        means=splats["means"].to(device),
        scales=splats["scales"].to(device),
        quats=splats["quats"].to(device),
        opacities=splats["opacities"].to(device),
        sh0=splats["sh0"].to(device),
        shN=splats["shN"].to(device),
        format="ply",
        save_to=str(save_to),
    )
    if not save_to.exists():
        raise RuntimeError(f"Checkpoint export failed: {save_to}")
    print(f"Exported Gaussian PLY from checkpoint: {save_to}")
    print(FINAL_GAUSSIAN_NOTE)
    return save_to


def find_latest_ply_candidate(train_dir):
    candidates = find_train_ply_candidates(train_dir)
    if not candidates:
        ckpts = find_gsplat_ckpt_candidates(train_dir) if "find_gsplat_ckpt_candidates" in globals() else sorted(Path(train_dir).glob("**/ckpt_*_rank*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
        if ckpts:
            fallback_path = Path(train_dir) / "ply" / "gaussians.ply"
            return export_gaussian_ply_from_checkpoint(ckpts[0], fallback_path)
        raise FileNotFoundError("No .ply candidates found in train_dir and no checkpoint fallback is available. Training/export likely did not produce a PLY.")
    explicit = [p for p in candidates if p.name == "gaussians.ply"]
    ordered = explicit + [p for p in candidates if p.name != "gaussians.ply"]
    for path in ordered:
        print(f"candidate: {path} ({path.stat().st_size / (1024*1024):.2f} MB)")
    print("Selected explicit gsplat Gaussian PLY candidate:" if explicit else "Selected latest PLY candidate:", ordered[0])
    return ordered[0]


def export_final_gaussian_ply(config):
    """Finalize metadata for result/gaussians.ply.

    Primary path: examples/da3_trainer.py writes config.result_ply_path directly via
    gsplat.export_splats(..., format="ply", save_to=config.result_ply_path).
    This function no longer copies arbitrary PLY candidates unless it must use a
    checkpoint fallback.
    """
    result_dir = Path(config.result_dir)
    result_dir.mkdir(parents=True, exist_ok=True)
    result = Path(config.result_ply_path)
    metadata_path = result_dir / "metadata.json"

    if not result.exists() or result.stat().st_size == 0:
        print("Final result/gaussians.ply is missing. Trying checkpoint fallback export.")
        ckpts = find_gsplat_ckpt_candidates(config.gsplat_train_dir) if "find_gsplat_ckpt_candidates" in globals() else sorted(Path(config.gsplat_train_dir).glob("**/ckpt_*_rank*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
        if ckpts:
            export_gaussian_ply_from_checkpoint(ckpts[0], result)
        else:
            raise FileNotFoundError(
                f"Missing final Gaussian PLY and no checkpoint fallback was found: {result}"
            )

    frames = json.loads((Path(config.output_dir) / "frames.json").read_text(encoding="utf-8"))
    metadata = {
        "input_video_path": str(config.input_video_path),
        "job_id": str(config.job_id),
        "frame_count": len(frames),
        "da3_repo_url": str(config.da3_repo_url),
        "da3_model_name": str(config.da3_model_name),
        "da3_checkpoint_or_hf_id": str(config.da3_checkpoint_or_hf_id),
        "gsplat_repo_url": str(config.gsplat_repo_url),
        "confidence_threshold": float(config.confidence_threshold),
        "gsplat_iterations": int(config.gsplat_iterations),
        "source_ply_path": str(result),
        "result_ply_path": str(result),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "note": FINAL_GAUSSIAN_NOTE,
        "export_note": "Primary export is written directly by examples/da3_trainer.py using gsplat.export_splats.",
        "blender_warning": "This PLY stores 3D Gaussian primitives. It is not a triangle mesh.",
    }
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Final Gaussian PLY is ready: {result}")
    print(f"Metadata: {metadata_path}")
    print(FINAL_GAUSSIAN_NOTE)
    return result, metadata_path

def package_and_download_result(config):
    result_dir = Path(config.result_dir)
    result_ply = Path(config.result_ply_path)
    metadata = result_dir / "metadata.json"
    cameras = Path(config.da3_output_dir) / "cameras.json"
    config_snapshot = result_dir / "config_snapshot.json"
    train_log = Path(config.gsplat_train_dir) / "train.log"
    config_snapshot.write_text(json.dumps(asdict(config), indent=2, ensure_ascii=False), encoding="utf-8")
    if not train_log.exists():
        train_log = result_dir / "train.log"
        train_log.write_text("train.log was not found at packaging time.\n", encoding="utf-8")
    for required in [result_ply, metadata, cameras, config_snapshot, train_log]:
        if not required.exists():
            raise FileNotFoundError(required)
    zip_path = result_dir / f"{config.job_id}_3dgs_gaussian_result.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(result_ply, "gaussians.ply")
        zf.write(metadata, "metadata.json")
        zf.write(cameras, "cameras.json")
        zf.write(config_snapshot, "config_snapshot.json")
        zf.write(train_log, "train.log")
    print(f"ZIP: {zip_path}")
    print(FINAL_GAUSSIAN_NOTE)
    if config.use_drive:
        print(f"Drive result dir: {result_dir}")
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Automatic download failed: {repr(exc)}")
        print(f"Download manually from: {zip_path}")
    return zip_path


debug_report = debug_pipeline_state(config)
final_ply_path, final_metadata_path = export_final_gaussian_ply(config)
result_zip_path = package_and_download_result(config)